<a href="https://colab.research.google.com/github/shobhis330/agentic-ai-debugging-test-automation/blob/main/agentic_ai_debugging_week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Agentic AI Software Debugging and Test Automation System


In [301]:

%pip install -q google-genai pytest


In [302]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [303]:

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with exactly: Gemini connection test successful."
)

print(response.text)


Gemini connection test successful.


In [304]:

buggy_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

print("Buggy code:")
print(buggy_code)


Buggy code:

def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [305]:

prompt = f"""
You are a Python debugging assistant.

Analyze the following code:
{buggy_code}

Identify:
1. The bug.
2. Why it occurs.
3. A corrected version of the code.
4. One test case to verify the correction.

Do not execute the code. Return a clear explanation.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)


Here is the analysis of the code:

### 1. The Bug
The function calculates a sample variance denominator ($n - 1$) instead of a standard arithmetic mean denominator ($n$), or it will crash with a `ZeroDivisionError` when given a list with a single element.

### 2. Why it Occurs
The code divides the `total` by `(len(numbers) - 1)` instead of `len(numbers)`. 
* **Mathematical error:** The formula for an average (mean) is the sum of the numbers divided by the count of the numbers (`len(numbers)`). Dividing by `len(numbers) - 1` calculates something else entirely (specifically, degrees of freedom used in sample variance).
* **Runtime error:** If the input list `numbers` contains only one element, `len(numbers)` is `1`. This results in `1 - 1 = 0`, causing a `ZeroDivisionError`. If the list is empty, it results in `-1`, which produces an incorrect mathematical result instead of failing gracefully.

### 3. A Corrected Version of the Code
```python
def calculate_average(numbers):
    if not nu

In [306]:

def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")

    total = sum(numbers)
    return total / len(numbers)


In [307]:

%%writefile test_debugger.py

import pytest
from math import isclose

def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")

    total = sum(numbers)
    return total / len(numbers)


def test_normal_average():
    assert isclose(calculate_average([2, 4, 6, 8]), 5.0)


def test_single_element():
    assert calculate_average([10]) == 10


def test_negative_numbers():
    assert calculate_average([-2, -4, -6]) == -4


def test_empty_list():
    with pytest.raises(ValueError):
        calculate_average([])


Overwriting test_debugger.py


In [308]:

!python -m pytest test_debugger.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/agentic-ai-debugging-test-automation
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.13, platformdirs-4.13.6
collected 4 items                                                              

test_debugger.py::test_normal_average PASSED                             [ 25%]
test_debugger.py::test_single_element PASSED                             [ 50%]
test_debugger.py::test_negative_numbers PASSED                           [ 75%]
test_debugger.py::test_empty_list PASSED                                 [100%]

============================== 4 passed in 0.12s ===============================


In [309]:

%%writefile test_average.py

import pytest
from solution import calculate_average


def test_normal_average():
    assert calculate_average([2, 4, 6, 8]) == 5.0


def test_single_element():
    assert calculate_average([10]) == 10


def test_negative_numbers():
    assert calculate_average([-2, -4, -6]) == -4


def test_empty_list():
    with pytest.raises(ValueError):
        calculate_average([])


Overwriting test_average.py


In [310]:

%%writefile solution.py

def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)


Overwriting solution.py


In [311]:

!python -m pytest test_average.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/agentic-ai-debugging-test-automation
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.13, platformdirs-4.13.6
collected 4 items                                                              

test_average.py::test_normal_average FAILED                              [ 25%]
test_average.py::test_single_element FAILED                              [ 50%]
test_average.py::test_negative_numbers FAILED                            [ 75%]
test_average.py::test_empty_list FAILED                                  [100%]

=================================== FAILURES ===================================
_____________________________ test_normal_average ______________________________

    def test_normal_average():
>       assert calculate_average([2, 4, 6, 8]) == 5.0
E       assert 6.6666666666666

In [312]:

import subprocess
from pathlib import Path
from google import genai
from google.colab import userdata

PROJECT_DIR = Path("/content")
SOLUTION_FILE = PROJECT_DIR / "solution.py"
TEST_FILE = PROJECT_DIR / "test_average.py"

# Read the original buggy code
buggy_code = SOLUTION_FILE.read_text()

# Run the tests and capture their failures
result = subprocess.run(
    ["python", "-m", "pytest", str(TEST_FILE), "-q"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    timeout=30
)

test_feedback = result.stdout + "\n" + result.stderr

print("Original test exit code:", result.returncode)
print("\nCaptured test feedback:\n")
print(test_feedback[:2500])


Original test exit code: 0

Captured test feedback:

...                                                                      [100%]
3 passed in 0.04s




In [313]:

prompt = f"""
You are a Python debugging agent.

Repair the code using the actual test failures.

ORIGINAL CODE:
{buggy_code}

TEST FAILURE OUTPUT:
{test_feedback}

Requirements:
- Keep the function name calculate_average.
- Calculate the correct arithmetic average.
- Raise ValueError for an empty list.
- Return the element for a single-element list.
- Do not change the tests.

Return only the complete Python code.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)


```python
def calculate_average(prices):
    if not prices:
        raise ValueError("The list cannot be empty.")
    return sum(prices) / len(prices)
```


In [314]:

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "/content/test_average.py", "-v"
    ],
    cwd="/content",
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("\nSUCCESS: All tests passed!")
else:
    print("\nREPAIR NOT VERIFIED: Some tests failed.")


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.13, platformdirs-4.13.6
collecting ... collected 3 items

test_average.py::test_normal_total PASSED                                [ 33%]
test_average.py::test_single_price PASSED                                [ 66%]
test_average.py::test_empty_prices PASSED                                [100%]

============================== 3 passed in 0.02s ===============================


SUCCESS: All tests passed!


In [315]:

from pathlib import Path
import subprocess
import sys
import re

# Project files
PROJECT_DIR = Path("/content")
SOLUTION_FILE = PROJECT_DIR / "solution.py"
TEST_FILE = PROJECT_DIR / "test_average.py"
BACKUP_FILE = PROJECT_DIR / "solution_original.py"

MAX_RETRIES = 3

# Preserve the initial buggy implementation once
if not BACKUP_FILE.exists():
    BACKUP_FILE.write_text(SOLUTION_FILE.read_text())

def run_tests():
    """Run the fixed test suite and return results."""
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            str(TEST_FILE), "-q"
        ],
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True,
        timeout=30
    )
    return result.returncode, result.stdout + "\n" + result.stderr


def extract_python_code(text):
    """Extract Python from a Gemini response."""
    match = re.search(
        r"```(?:python)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    code = match.group(1).strip() if match else text.strip()
    compile(code, "candidate_solution.py", "exec")
    return code


def request_repair(code, feedback):
    """Ask Gemini for a candidate repair."""
    prompt = f"""
You are a Python debugging agent.

Repair the function using the supplied test results.

CURRENT CODE:
{code}

TEST RESULTS:
{feedback}

Requirements:
- Preserve the function name calculate_average.
- Return the arithmetic mean for a non-empty list.
- Raise ValueError for an empty list.
- Do not change or rewrite the tests.
- Return only the complete Python implementation.
"""
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response.")

    return extract_python_code(response.text)


def run_debugging_agent():
    original_code = BACKUP_FILE.read_text()
    SOLUTION_FILE.write_text(original_code)

    for attempt in range(MAX_RETRIES + 1):
        code = SOLUTION_FILE.read_text()
        exit_code, feedback = run_tests()

        print(f"\n--- Test run {attempt + 1} ---")
        print(feedback[-1800:])

        if exit_code == 0:
            print("\nSUCCESS: All tests passed.")
            return True

        if attempt == MAX_RETRIES:
            break

        print(f"\nRepair attempt {attempt + 1} of {MAX_RETRIES}")

        try:
            candidate = request_repair(code, feedback)

            # Basic structural validation before saving
            if "def calculate_average(numbers):" not in candidate:
                raise ValueError(
                    "Candidate does not contain the expected function."
                )

            SOLUTION_FILE.write_text(candidate)
            print("Candidate saved. Running tests again...")

        except Exception as error:
            print("Repair attempt could not be applied:", error)

    print("\nSTOPPED: Retry limit reached; tests remain failing.")
    return False


print("Debugging agent is ready.")


Debugging agent is ready.


In [316]:

success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")



--- Test run 1 ---

==================================== ERRORS ====================================
_______________________ ERROR collecting test_average.py _______________________
ImportError while importing test module '/content/test_average.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
test_average.py:1: in <module>
    from solution import calculate_total
E   ImportError: cannot import name 'calculate_total' from 'solution' (/content/solution.py)
=========================== short test summary info ============================
ERROR test_average.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.23s



Repair attempt 1 of 3
Candidate saved. Running tests again...

--- Test run 2 ---

===========================

In [317]:

from pathlib import Path

solution_file = Path("/content/solution.py")
backup_file = Path("/content/solution_original.py")

# Restore the original code for a controlled experiment
solution_file.write_text(backup_file.read_text())

print("Original buggy solution restored.")
print("\nCurrent solution:\n")
print(solution_file.read_text())


Original buggy solution restored.

Current solution:

def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [318]:

from pathlib import Path

solution_file = Path("/content/solution.py")

buggy_code = '''def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
'''

solution_file.write_text(buggy_code)

print("Buggy solution prepared:")
print(solution_file.read_text())


Buggy solution prepared:
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [319]:
success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")


--- Test run 1 ---

==================================== ERRORS ====================================
_______________________ ERROR collecting test_average.py _______________________
ImportError while importing test module '/content/test_average.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
test_average.py:1: in <module>
    from solution import calculate_total
E   ImportError: cannot import name 'calculate_total' from 'solution' (/content/solution.py)
=========================== short test summary info ============================
ERROR test_average.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.13s



Repair attempt 1 of 3
Candidate saved. Running tests again...

--- Test run 2 ---

===========================

In [320]:

from pathlib import Path

PROJECT_DIR = Path("/content")
SOLUTION_FILE = PROJECT_DIR / "solution.py"
BACKUP_FILE = PROJECT_DIR / "solution_original.py"

# Explicitly set the original buggy implementation as the baseline
buggy_code = '''def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
'''

BACKUP_FILE.write_text(buggy_code)
SOLUTION_FILE.write_text(buggy_code)

print("Baseline reset.")
print("\nBackup file:")
print(BACKUP_FILE.read_text())

print("\nCurrent solution:")
print(SOLUTION_FILE.read_text())


Baseline reset.

Backup file:
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)


Current solution:
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [321]:

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "/content/test_average.py", "-q"],
    cwd="/content",
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout)
print("Exit code:", result.returncode)



==================================== ERRORS ====================================
_______________________ ERROR collecting test_average.py _______________________
ImportError while importing test module '/content/test_average.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
test_average.py:1: in <module>
    from solution import calculate_total
E   ImportError: cannot import name 'calculate_total' from 'solution' (/content/solution.py)
=========================== short test summary info ============================
ERROR test_average.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.21s

Exit code: 2


In [322]:
success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")


--- Test run 1 ---

==================================== ERRORS ====================================
_______________________ ERROR collecting test_average.py _______________________
ImportError while importing test module '/content/test_average.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
test_average.py:1: in <module>
    from solution import calculate_total
E   ImportError: cannot import name 'calculate_total' from 'solution' (/content/solution.py)
=========================== short test summary info ============================
ERROR test_average.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.48s



Repair attempt 1 of 3
Candidate saved. Running tests again...

--- Test run 2 ---

===========================

In [323]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

columns = [
    "experiment_id",
    "bug_type",
    "initial_tests_failed",
    "repair_attempts",
    "final_tests_passed",
    "final_status",
    "notes"
]

row = {
    "experiment_id": "EXP001",
    "bug_type": "Off-by-one error and missing empty-input validation",
    "initial_tests_failed": 4,
    "repair_attempts": 1,
    "final_tests_passed": 4,
    "final_status": "PASS",
    "notes": "Gemini-generated repair verified by pytest"
}

write_header = not results_file.exists()

with open(results_file, "a", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=columns)

    if write_header:
        writer.writeheader()

    writer.writerow(row)

print("Experiment result saved to:", results_file)
print(results_file.read_text())


Experiment result saved to: /content/experiment_results.csv
experiment_id,bug_type,initial_tests_failed,repair_attempts,final_tests_passed,final_status,notes
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair
EXP003,Undefined variable (NameError),3,1,3,PASS,Gemini-generated repair verified by pytest
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair
EXP003,Undefined variable (NameError),3,1,3,PASS,Gemini-generated repair verified by pytest
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest



In [324]:

from pathlib import Path

Path("/content/solution.py").write_text('''def is_adult(age):
    return age > 18
''')

Path("/content/test_average.py").write_text('''import pytest
from solution import is_adult

def test_adult_age():
    assert is_adult(25) is True

def test_exactly_18():
    assert is_adult(18) is True

def test_under_18():
    assert is_adult(17) is False
''')

print("Second bug case prepared: incorrect conditional logic.")


Second bug case prepared: incorrect conditional logic.


In [325]:

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "/content/test_average.py", "-q"],
    cwd="/content",
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout)
print("Exit code:", result.returncode)


.F.                                                                      [100%]
=================================== FAILURES ===================================
_______________________________ test_exactly_18 ________________________________

    def test_exactly_18():
>       assert is_adult(18) is True
E       assert False is True
E        +  where False = is_adult(18)

test_average.py:8: AssertionError
=========================== short test summary info ============================
FAILED test_average.py::test_exactly_18 - assert False is True
1 failed, 2 passed in 0.05s

Exit code: 1


In [326]:

def request_repair(code, feedback):
    """Ask Gemini to repair a general Python function."""

    prompt = f"""
You are an autonomous Python debugging agent.

Repair the provided Python code using the test failure feedback.

CURRENT CODE:
{code}

TEST FAILURE OUTPUT:
{feedback}

Requirements:
- Identify the actual bug from the code and test feedback.
- Preserve the intended function names and behavior.
- Do not modify the test file.
- Return the complete corrected Python implementation only.
- Do not include Markdown code fences or explanations.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response.")

    return extract_python_code(response.text)


In [327]:

def run_debugging_agent():
    """Run tests and attempt up to MAX_RETRIES repairs."""

    # Start from the current solution, not an old backup.
    for attempt in range(MAX_RETRIES + 1):
        code = SOLUTION_FILE.read_text()
        exit_code, feedback = run_tests()

        print(f"\n--- Test run {attempt + 1} ---")
        print(feedback[-1500:])

        if exit_code == 0:
            print("\nSUCCESS: All tests passed.")
            return True

        if attempt == MAX_RETRIES:
            break

        print(f"\nRepair attempt {attempt + 1} of {MAX_RETRIES}")

        try:
            candidate = request_repair(code, feedback)

            # Validate syntax before saving
            compile(candidate, "candidate_solution.py", "exec")

            # Save candidate and let pytest determine whether it works
            SOLUTION_FILE.write_text(candidate)
            print("Candidate saved. Running tests again...")

        except Exception as error:
            print("Repair attempt failed:", error)

    print("\nSTOPPED: Retry limit reached or repair attempts failed.")
    return False


print("Generic debugging agent is ready.")


Generic debugging agent is ready.


In [328]:

success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")



--- Test run 1 ---
.F.                                                                      [100%]
=================================== FAILURES ===================================
_______________________________ test_exactly_18 ________________________________

    def test_exactly_18():
>       assert is_adult(18) is True
E       assert False is True
E        +  where False = is_adult(18)

test_average.py:8: AssertionError
=========================== short test summary info ============================
FAILED test_average.py::test_exactly_18 - assert False is True
1 failed, 2 passed in 0.06s



Repair attempt 1 of 3
Candidate saved. Running tests again...

--- Test run 2 ---
...                                                                      [100%]
3 passed in 0.01s



SUCCESS: All tests passed.

Final status: PASS


In [329]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

row = {
    "experiment_id": "EXP002",
    "bug_type": "Incorrect conditional logic",
    "initial_tests_failed": 1,
    "repair_attempts": 1,
    "final_tests_passed": 3,
    "final_status": "PASS",
    "notes": "Gemini changed the age boundary condition; pytest verified the repair"
}

with open(results_file, "a", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "experiment_id",
            "bug_type",
            "initial_tests_failed",
            "repair_attempts",
            "final_tests_passed",
            "final_status",
            "notes"
        ]
    )
    writer.writerow(row)

print("Second experiment recorded.")
print(results_file.read_text())


Second experiment recorded.
experiment_id,bug_type,initial_tests_failed,repair_attempts,final_tests_passed,final_status,notes
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair
EXP003,Undefined variable (NameError),3,1,3,PASS,Gemini-generated repair verified by pytest
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair
EXP003,Undefined variable (NameError),3,1,3,PASS,Gemini-generated repair verified by pytest
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair



In [330]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

with open(results_file, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

total = len(rows)
successful = sum(row["final_status"] == "PASS" for row in rows)
total_repairs = sum(int(row["repair_attempts"]) for row in rows)
total_initial_failures = sum(
    int(row["initial_tests_failed"]) for row in rows
)
total_final_passes = sum(
    int(row["final_tests_passed"]) for row in rows
)

success_rate = (successful / total * 100) if total else 0

print("=== AGENT EVALUATION SUMMARY ===")
print("Experiments:", total)
print("Successful experiments:", successful)
print(f"Experiment success rate: {success_rate:.1f}%")
print("Total repair attempts:", total_repairs)
print("Initial test failures:", total_initial_failures)
print("Final passing test counts:", total_final_passes)

print("\n=== INDIVIDUAL RESULTS ===")
for row in rows:
    print(
        f'{row["experiment_id"]}: {row["bug_type"]} | '
        f'{row["final_status"]} | '
        f'{row["repair_attempts"]} repair attempt(s)'
    )


=== AGENT EVALUATION SUMMARY ===
Experiments: 8
Successful experiments: 8
Experiment success rate: 100.0%
Total repair attempts: 8
Initial test failures: 21
Final passing test counts: 27

=== INDIVIDUAL RESULTS ===
EXP001: Off-by-one error and missing empty-input validation | PASS | 1 repair attempt(s)
EXP002: Incorrect conditional logic | PASS | 1 repair attempt(s)
EXP003: Undefined variable (NameError) | PASS | 1 repair attempt(s)
EXP001: Off-by-one error and missing empty-input validation | PASS | 1 repair attempt(s)
EXP002: Incorrect conditional logic | PASS | 1 repair attempt(s)
EXP003: Undefined variable (NameError) | PASS | 1 repair attempt(s)
EXP001: Off-by-one error and missing empty-input validation | PASS | 1 repair attempt(s)
EXP002: Incorrect conditional logic | PASS | 1 repair attempt(s)


In [331]:

from pathlib import Path

Path("/content/solution.py").write_text('''def calculate_total(prices):
    total = sum(price for price in prices)
    return tota
''')

Path("/content/test_average.py").write_text('''from solution import calculate_total

def test_normal_total():
    assert calculate_total([10, 20, 30]) == 60

def test_single_price():
    assert calculate_total([15]) == 15

def test_empty_prices():
    assert calculate_total([]) == 0
''')

print("Third bug case prepared: undefined variable.")


Third bug case prepared: undefined variable.


In [332]:

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "/content/test_average.py", "-q"],
    cwd="/content",
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout)
print("Exit code:", result.returncode)


FFF                                                                      [100%]
=================================== FAILURES ===================================
______________________________ test_normal_total _______________________________

    def test_normal_total():
>       assert calculate_total([10, 20, 30]) == 60
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

test_average.py:4: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

prices = [10, 20, 30]

    def calculate_total(prices):
        total = sum(price for price in prices)
>       return tota
               ^^^^
E       NameError: name 'tota' is not defined

solution.py:3: NameError
______________________________ test_single_price _______________________________

    def test_single_price():
>       assert calculate_total([15]) == 15
               ^^^^^^^^^^^^^^^^^^^^^

test_average.py:7: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

prices = [15]

    d

In [333]:
success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")


--- Test run 1 ---
:
        total = sum(price for price in prices)
>       return tota
               ^^^^
E       NameError: name 'tota' is not defined

solution.py:3: NameError
______________________________ test_single_price _______________________________

    def test_single_price():
>       assert calculate_total([15]) == 15
               ^^^^^^^^^^^^^^^^^^^^^

test_average.py:7: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

prices = [15]

    def calculate_total(prices):
        total = sum(price for price in prices)
>       return tota
               ^^^^
E       NameError: name 'tota' is not defined

solution.py:3: NameError
______________________________ test_empty_prices _______________________________

    def test_empty_prices():
>       assert calculate_total([]) == 0
               ^^^^^^^^^^^^^^^^^^^

test_average.py:10: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

prices = []

    def calcula

In [334]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

row = {
    "experiment_id": "EXP003",
    "bug_type": "Undefined variable (NameError)",
    "initial_tests_failed": 3,
    "repair_attempts": 1,
    "final_tests_passed": 3,
    "final_status": "PASS",
    "notes": "Gemini-generated repair verified by pytest"
}

with open(results_file, "a", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "experiment_id",
            "bug_type",
            "initial_tests_failed",
            "repair_attempts",
            "final_tests_passed",
            "final_status",
            "notes"
        ]
    )
    writer.writerow(row)

print("Experiment 3 recorded successfully.")
print(results_file.read_text())


Experiment 3 recorded successfully.
experiment_id,bug_type,initial_tests_failed,repair_attempts,final_tests_passed,final_status,notes
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair
EXP003,Undefined variable (NameError),3,1,3,PASS,Gemini-generated repair verified by pytest
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair
EXP003,Undefined variable (NameError),3,1,3,PASS,Gemini-generated repair verified by pytest
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the re

In [335]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

with open(results_file, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

total = len(rows)
successful = sum(r["final_status"] == "PASS" for r in rows)
repairs = sum(int(r["repair_attempts"]) for r in rows)
initial_failures = sum(int(r["initial_tests_failed"]) for r in rows)
final_passes = sum(int(r["final_tests_passed"]) for r in rows)

print("=== WEEK 2 EVALUATION SUMMARY ===")
print("Experiments:", total)
print("Successful experiments:", successful)
print(f"Experiment success rate: {successful / total * 100:.1f}%" if total else "Experiment success rate: N/A")
print("Repair attempts:", repairs)
print("Initial test failures:", initial_failures)
print("Final passing test counts:", final_passes)

print("\n=== BUG TYPES TESTED ===")
for row in rows:
    print(f'{row["experiment_id"]}: {row["bug_type"]} -> {row["final_status"]}')


=== WEEK 2 EVALUATION SUMMARY ===
Experiments: 9
Successful experiments: 9
Experiment success rate: 100.0%
Repair attempts: 9
Initial test failures: 24
Final passing test counts: 30

=== BUG TYPES TESTED ===
EXP001: Off-by-one error and missing empty-input validation -> PASS
EXP002: Incorrect conditional logic -> PASS
EXP003: Undefined variable (NameError) -> PASS
EXP001: Off-by-one error and missing empty-input validation -> PASS
EXP002: Incorrect conditional logic -> PASS
EXP003: Undefined variable (NameError) -> PASS
EXP001: Off-by-one error and missing empty-input validation -> PASS
EXP002: Incorrect conditional logic -> PASS
EXP003: Undefined variable (NameError) -> PASS


In [336]:

import json
from pathlib import Path

bug_cases = [
    {
        "id": "TC01",
        "bug_type": "Off-by-one error",
        "function_name": "calculate_average",
        "buggy_code": """def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
""",
        "tests": [
            "Normal average",
            "Single-element list",
            "Negative numbers",
            "Empty-list validation"
        ]
    },
    {
        "id": "TC02",
        "bug_type": "Incorrect conditional logic",
        "function_name": "is_adult",
        "buggy_code": """def is_adult(age):
    return age > 18
""",
        "tests": [
            "Adult age",
            "Exactly 18",
            "Under 18"
        ]
    },
    {
        "id": "TC03",
        "bug_type": "Undefined variable",
        "function_name": "calculate_total",
        "buggy_code": """def calculate_total(prices):
    total = sum(price for price in prices)
    return tota
""",
        "tests": [
            "Normal total",
            "Single price",
            "Empty prices"
        ]
    }
]

dataset_path = Path("/content/bug_cases.json")
dataset_path.write_text(
    json.dumps(bug_cases, indent=2),
    encoding="utf-8"
)

print("Dataset created:", dataset_path)
print("Bug cases:", len(bug_cases))
print(dataset_path.read_text())


Dataset created: /content/bug_cases.json
Bug cases: 3
[
  {
    "id": "TC01",
    "bug_type": "Off-by-one error",
    "function_name": "calculate_average",
    "buggy_code": "def calculate_average(numbers):\n    total = sum(numbers)\n    return total / (len(numbers) - 1)\n",
    "tests": [
      "Normal average",
      "Single-element list",
      "Negative numbers",
      "Empty-list validation"
    ]
  },
  {
    "id": "TC02",
    "bug_type": "Incorrect conditional logic",
    "function_name": "is_adult",
    "buggy_code": "def is_adult(age):\n    return age > 18\n",
    "tests": [
      "Adult age",
      "Exactly 18",
      "Under 18"
    ]
  },
  {
    "id": "TC03",
    "bug_type": "Undefined variable",
    "function_name": "calculate_total",
    "buggy_code": "def calculate_total(prices):\n    total = sum(price for price in prices)\n    return tota\n",
    "tests": [
      "Normal total",
      "Single price",
      "Empty prices"
    ]
  }
]


In [337]:

import json
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content")
DATASET_FILE = PROJECT_DIR / "bug_cases.json"
CASES_DIR = PROJECT_DIR / "evaluation_cases"

CASES_DIR.mkdir(exist_ok=True)

with open(DATASET_FILE, encoding="utf-8") as file:
    bug_cases = json.load(file)

# Fixed tests are written independently of Gemini's output.
test_definitions = {
    "TC01": '''import pytest
from solution import calculate_average

def test_normal_average():
    assert calculate_average([2, 4, 6, 8]) == 5.0

def test_single_element():
    assert calculate_average([10]) == 10

def test_negative_numbers():
    assert calculate_average([-2, -4, -6]) == -4

def test_empty_list():
    with pytest.raises(ValueError):
        calculate_average([])
''',
    "TC02": '''from solution import is_adult

def test_adult_age():
    assert is_adult(25) is True

def test_exactly_18():
    assert is_adult(18) is True

def test_under_18():
    assert is_adult(17) is False
''',
    "TC03": '''from solution import calculate_total

def test_normal_total():
    assert calculate_total([10, 20, 30]) == 60

def test_single_price():
    assert calculate_total([15]) == 15

def test_empty_prices():
    assert calculate_total([]) == 0
'''
}

for case in bug_cases:
    case_dir = CASES_DIR / case["id"]
    case_dir.mkdir(exist_ok=True)

    (case_dir / "solution.py").write_text(
        case["buggy_code"], encoding="utf-8"
    )
    (case_dir / "test_solution.py").write_text(
        test_definitions[case["id"]], encoding="utf-8"
    )

    # Run from this case's directory to avoid importing another solution.py.
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "--tb=short", "test_solution.py"
        ],
        cwd=str(case_dir),
        capture_output=True,
        text=True,
        timeout=30
    )

    print(f'\n{case["id"]}: {case["bug_type"]}')
    print(result.stdout)
    print("Initial exit code:", result.returncode)

print("\nEvaluation case folders prepared.")



TC01: Off-by-one error
FFFF                                                                     [100%]
=================================== FAILURES ===================================
_____________________________ test_normal_average ______________________________
test_solution.py:5: in test_normal_average
    assert calculate_average([2, 4, 6, 8]) == 5.0
E   assert 6.666666666666667 == 5.0
E    +  where 6.666666666666667 = calculate_average([2, 4, 6, 8])
_____________________________ test_single_element ______________________________
test_solution.py:8: in test_single_element
    assert calculate_average([10]) == 10
           ^^^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in calculate_average
    return total / (len(numbers) - 1)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
E   ZeroDivisionError: division by zero
____________________________ test_negative_numbers _____________________________
test_solution.py:11: in test_negative_numbers
    assert calculate_average([-2, -4, -6]) == -4
E   assert 

In [338]:

import csv
import json
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content")
CASES_DIR = PROJECT_DIR / "evaluation_cases"
DATASET_FILE = PROJECT_DIR / "bug_cases.json"
RUN_RESULTS = PROJECT_DIR / "evaluation_runner_results.csv"

with open(DATASET_FILE, encoding="utf-8") as f:
    cases = json.load(f)


def run_case_tests(case_dir):
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "--tb=short", "test_solution.py"
        ],
        cwd=str(case_dir),
        capture_output=True,
        text=True,
        timeout=30
    )
    return result.returncode, result.stdout + "\n" + result.stderr


def ask_gemini_for_fix(code, feedback, function_name):
    prompt = f"""
You are a Python debugging agent.

Repair this code based on the test failures.

CURRENT CODE:
{code}

TEST FAILURE FEEDBACK:
{feedback}

Requirements:
- Preserve the function name: {function_name}
- Fix the demonstrated bug.
- Preserve behavior required by the tests.
- Do not change the tests.
- Return only complete Python code, preferably in a Python code block.
"""
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response.")

    return extract_python_code(response.text)


results = []

for case in cases:
    case_id = case["id"]
    case_dir = CASES_DIR / case_id
    solution_file = case_dir / "solution.py"

    # Reset to the exact dataset baseline for a reproducible run.
    original_code = case["buggy_code"]
    solution_file.write_text(original_code, encoding="utf-8")

    attempts = 0
    final_status = "FAIL"
    final_output = ""

    print(f"\n{'=' * 55}")
    print(f'{case_id}: {case["bug_type"]}')

    try:
        for run_number in range(4):  # Initial test + at most 3 repairs
            exit_code, feedback = run_case_tests(case_dir)
            final_output = feedback

            print(f"\nTest run {run_number + 1}; exit code: {exit_code}")
            print(feedback[-900:])

            if exit_code == 0:
                final_status = "PASS"
                break

            if attempts >= 3:
                break

            attempts += 1
            print(f"Gemini repair attempt {attempts}/3")

            candidate = ask_gemini_for_fix(
                solution_file.read_text(encoding="utf-8"),
                feedback,
                case["function_name"]
            )

            # Syntax-check the candidate before saving it.
            compile(candidate, f"{case_id}_candidate.py", "exec")
            solution_file.write_text(candidate, encoding="utf-8")

    except Exception as error:
        final_output = str(error)
        print("Case error:", error)

    results.append({
        "experiment_id": case_id,
        "bug_type": case["bug_type"],
        "repair_attempts": attempts,
        "final_status": final_status,
        "details": final_output[-1000:].replace("\n", " ")
    })

    print("Final status:", final_status)

with open(RUN_RESULTS, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "experiment_id",
            "bug_type",
            "repair_attempts",
            "final_status",
            "details"
        ]
    )
    writer.writeheader()
    writer.writerows(results)

print("\n=== EVALUATION RUNNER SUMMARY ===")
print("Cases:", len(results))
print("Passed:", sum(r["final_status"] == "PASS" for r in results))
print("Failed:", sum(r["final_status"] == "FAIL" for r in results))
print("Results saved to:", RUN_RESULTS)



TC01: Off-by-one error

Test run 1; exit code: 1
o
____________________________ test_negative_numbers _____________________________
test_solution.py:11: in test_negative_numbers
    assert calculate_average([-2, -4, -6]) == -4
E   assert -6.0 == -4
E    +  where -6.0 = calculate_average([-2, -4, -6])
_______________________________ test_empty_list ________________________________
test_solution.py:14: in test_empty_list
    with pytest.raises(ValueError):
         ^^^^^^^^^^^^^^^^^^^^^^^^^
E   Failed: DID NOT RAISE <class 'ValueError'>
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_average - assert 6.666666666666667 == 5.0
FAILED test_solution.py::test_single_element - ZeroDivisionError: division by...
FAILED test_solution.py::test_negative_numbers - assert -6.0 == -4
FAILED test_solution.py::test_empty_list - Failed: DID NOT RAISE <class 'Valu...
4 failed in 0.05s


Gemini repair attempt 1/3

Test run 2; exit code:

In [339]:
import pandas as pd

results = pd.read_csv("/content/evaluation_runner_results.csv")
display(results)

,experiment_id,bug_type,repair_attempts,final_status,details
0,TC01,Off-by-one error,1,PASS,.... ...
1,TC02,Incorrect conditional logic,1,PASS,... ...
2,TC03,Undefined variable,1,PASS,... ...


In [340]:
total = len(results)
passed = (results["final_status"] == "PASS").sum()
failed = (results["final_status"] == "FAIL").sum()

print("Total cases:", total)
print("Passed:", passed)
print("Failed:", failed)
print("Success rate:", round(passed / total * 100, 2), "%")
print("Total repair attempts:", results["repair_attempts"].sum())

Total cases: 3
Passed: 3
Failed: 0
Success rate: 100.0 %
Total repair attempts: 3


In [341]:
import json

def create_agent_result(
    case_id,
    bug_type,
    status,
    repair_attempts,
    message
):
    result = {
        "case_id": case_id,
        "bug_type": bug_type,
        "status": status,
        "repair_attempts": repair_attempts,
        "message": message
    }

    return json.dumps(result, indent=2)

print(create_agent_result(
    case_id="TC01",
    bug_type="Off-by-one error",
    status="PASS",
    repair_attempts=1,
    message="All tests passed after repair."
))

{
  "case_id": "TC01",
  "bug_type": "Off-by-one error",
  "status": "PASS",
  "repair_attempts": 1,
  "message": "All tests passed after repair."
}


In [342]:
sample_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

sample_feedback = """
The average is incorrect.
A single-element list causes division by zero.
An empty list should raise ValueError.
"""

try:
    result = get_structured_repair(
        sample_code,
        sample_feedback,
        "calculate_average"
    )

    print(json.dumps(result, indent=2))

except Exception as e:
    print("Error:", type(e).__name__, str(e))

{
  "bug_type": "ZeroDivisionError and incorrect arithmetic logic",
  "explanation": "The original code uses `len(numbers) - 1` in the denominator instead of `len(numbers)`, which results in a division by zero error for single-element lists, incorrect averages for all lists, and a ZeroDivisionError for empty lists instead of raising a ValueError.",
  "fixed_code": "def calculate_average(numbers):\n    if not numbers:\n        raise ValueError(\"List is empty\")\n    total = sum(numbers)\n    return total / len(numbers)",
  "confidence": 1
}


In [343]:
import json

def get_structured_repair(code, test_feedback, function_name):
    prompt = f"""
You are a Python debugging agent.

Analyze the buggy code and test failures.

Function name: {function_name}

Buggy code:
{code}

Test feedback:
{test_feedback}

Return a JSON object with exactly these fields:
- bug_type: string
- explanation: string
- fixed_code: string
- confidence: number between 0 and 1

Return valid JSON only. Do not use Markdown code fences.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )

    result = json.loads(response.text)

    required_fields = {
        "bug_type",
        "explanation",
        "fixed_code",
        "confidence"
    }

    if not required_fields.issubset(result):
        raise ValueError("Missing required JSON fields.")

    if not isinstance(result["fixed_code"], str):
        raise ValueError("fixed_code must be a string.")

    if not isinstance(result["confidence"], (int, float)):
        raise ValueError("Confidence must be numeric.")

    if not 0 <= result["confidence"] <= 1:
        raise ValueError("Confidence must be between 0 and 1.")

    compile(result["fixed_code"], "<gemini_candidate>", "exec")

    return result

print("Structured repair function defined successfully!")

Structured repair function defined successfully!


In [344]:
sample_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

sample_feedback = """
1. calculate_average([2, 4, 6, 8]) should return 5.0.
2. A single-element list should return its element.
3. Negative numbers should be averaged correctly.
4. An empty list should raise ValueError.
"""

try:
    result = get_structured_repair(
        sample_code,
        sample_feedback,
        "calculate_average"
    )

    print(json.dumps(result, indent=2))

except Exception as e:
    print("Error:", type(e).__name__, str(e))

{
  "bug_type": "Logic Error",
  "explanation": "The function incorrectly divides the sum by (len(numbers) - 1) instead of the total count of elements len(numbers). This also causes a ZeroDivisionError for single-element lists and negative results or incorrect averages otherwise. Additionally, it does not handle empty lists by raising a ValueError.",
  "fixed_code": "def calculate_average(numbers):\n    if not numbers:\n        raise ValueError(\"List is empty\")\n    return sum(numbers) / len(numbers)",
  "confidence": 1.0
}


In [345]:
from pathlib import Path
import subprocess
import sys

solution_path = Path("/content/evaluation_cases/TC01/solution.py")

# Save Gemini's generated code into the isolated TC01 case.
solution_path.write_text(result["fixed_code"], encoding="utf-8")

# Run the existing TC01 test suite.
test_run = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "-q", "--tb=short", "test_solution.py"
    ],
    cwd="/content/evaluation_cases/TC01",
    capture_output=True,
    text=True,
    timeout=30
)

print(test_run.stdout)
print(test_run.stderr)
print("Final test status:", "PASS" if test_run.returncode == 0 else "FAIL")

....                                                                     [100%]
4 passed in 0.01s


Final test status: PASS


In [346]:
print("run_debugging_agent exists:", "run_debugging_agent" in globals())
print("get_structured_repair exists:", "get_structured_repair" in globals())

run_debugging_agent exists: True
get_structured_repair exists: True


In [347]:
import inspect

print(inspect.getsource(run_debugging_agent))

def run_debugging_agent():
    """Run tests and attempt up to MAX_RETRIES repairs."""

    # Start from the current solution, not an old backup.
    for attempt in range(MAX_RETRIES + 1):
        code = SOLUTION_FILE.read_text()
        exit_code, feedback = run_tests()

        print(f"\n--- Test run {attempt + 1} ---")
        print(feedback[-1500:])

        if exit_code == 0:
            print("\nSUCCESS: All tests passed.")
            return True

        if attempt == MAX_RETRIES:
            break

        print(f"\nRepair attempt {attempt + 1} of {MAX_RETRIES}")

        try:
            candidate = request_repair(code, feedback)

            # Validate syntax before saving
            compile(candidate, "candidate_solution.py", "exec")

            # Save candidate and let pytest determine whether it works
            SOLUTION_FILE.write_text(candidate)
            print("Candidate saved. Running tests again...")

        except Exception as error:
            print("Repair a

In [348]:

def run_debugging_agent():
    """Run tests and repair code using Gemini's structured JSON output."""

    for attempt in range(MAX_RETRIES + 1):
        code = SOLUTION_FILE.read_text(encoding="utf-8")
        exit_code, feedback = run_tests()

        print(f"\n--- Test run {attempt + 1} ---")
        print(feedback[-1500:])

        if exit_code == 0:
            print("\nSUCCESS: All tests passed.")
            return True

        if attempt == MAX_RETRIES:
            break

        print(f"\nRepair attempt {attempt + 1} of {MAX_RETRIES}")

        try:
            # Extract the function name from the current code.
            import ast

            tree = ast.parse(code)
            functions = [
                node.name for node in tree.body
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            ]

            if not functions:
                raise ValueError("No function definition found in solution.")

            function_name = functions[0]

            # Ask Gemini for structured JSON repair.
            repair = get_structured_repair(
                code,
                feedback,
                function_name
            )

            print("Detected bug:", repair["bug_type"])
            print("Explanation:", repair["explanation"])
            print("Confidence:", repair["confidence"])

            candidate = repair["fixed_code"]

            # Validate syntax before saving.
            compile(candidate, "candidate_solution.py", "exec")

            # Save and verify with pytest on the next iteration.
            SOLUTION_FILE.write_text(candidate, encoding="utf-8")
            print("Candidate saved. Running tests again...")

        except Exception as error:
            print("Repair attempt failed:", error)

    print("\nSTOPPED: Retry limit reached or repair attempts failed.")
    return False


In [349]:
print("Solution file:", SOLUTION_FILE)
print("Test file:", TEST_FILE)
print("Maximum repairs:", MAX_RETRIES)
print("Solution exists:", SOLUTION_FILE.exists())
print("Test file exists:", TEST_FILE.exists())

Solution file: /content/solution.py
Test file: /content/test_average.py
Maximum repairs: 3
Solution exists: True
Test file exists: True


In [350]:
run_debugging_agent()


--- Test run 1 ---
...                                                                      [100%]
3 passed in 0.01s



SUCCESS: All tests passed.


True

In [351]:
from pathlib import Path

tc01_dir = Path("/content/evaluation_cases/TC01")
tc01_solution = tc01_dir / "solution.py"

# Save the known buggy version for this controlled test.
buggy_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

tc01_solution.write_text(buggy_code.strip() + "\n", encoding="utf-8")

# Point the existing agent at TC01 and its tests.
SOLUTION_FILE = tc01_solution
TEST_FILE = tc01_dir / "test_solution.py"

print("Solution:", SOLUTION_FILE)
print("Tests:", TEST_FILE)
print("Both files exist:", SOLUTION_FILE.exists() and TEST_FILE.exists())

Solution: /content/evaluation_cases/TC01/solution.py
Tests: /content/evaluation_cases/TC01/test_solution.py
Both files exist: True


In [352]:
run_debugging_agent()


--- Test run 1 ---

>       assert calculate_average([10]) == 10
               ^^^^^^^^^^^^^^^^^^^^^^^

evaluation_cases/TC01/test_solution.py:8: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

numbers = [10]

    def calculate_average(numbers):
        total = sum(numbers)
>       return total / (len(numbers) - 1)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
E       ZeroDivisionError: division by zero

evaluation_cases/TC01/solution.py:3: ZeroDivisionError
____________________________ test_negative_numbers _____________________________

    def test_negative_numbers():
>       assert calculate_average([-2, -4, -6]) == -4
E       assert -6.0 == -4
E        +  where -6.0 = calculate_average([-2, -4, -6])

evaluation_cases/TC01/test_solution.py:11: AssertionError
_______________________________ test_empty_list ________________________________

    def test_empty_list():
>       with pytest.raises(ValueError):
             ^^^^^^^^^^^^^^^^^^^^^^^^^
E     

True

In [353]:
print("Maximum repair attempts:", MAX_RETRIES)
print("Agent function available:", callable(run_debugging_agent))
print("Structured repair available:", callable(get_structured_repair))

Maximum repair attempts: 3
Agent function available: True
Structured repair available: True


In [354]:
from unittest.mock import patch

with patch(
    "__main__.run_tests",
    return_value=(1, "Simulated test failure")
):
    with patch(
        "__main__.get_structured_repair",
        side_effect=ValueError("Simulated invalid repair response")
    ):
        result = run_debugging_agent()

print("\nExpected result: False")
print("Actual result:", result)
print("Retry limit:", MAX_RETRIES)


--- Test run 1 ---
Simulated test failure

Repair attempt 1 of 3
Repair attempt failed: Simulated invalid repair response

--- Test run 2 ---
Simulated test failure

Repair attempt 2 of 3
Repair attempt failed: Simulated invalid repair response

--- Test run 3 ---
Simulated test failure

Repair attempt 3 of 3
Repair attempt failed: Simulated invalid repair response

--- Test run 4 ---
Simulated test failure

STOPPED: Retry limit reached or repair attempts failed.

Expected result: False
Actual result: False
Retry limit: 3


In [355]:
print("Real pytest runner:", run_tests.__module__)
print("Solution file:", SOLUTION_FILE)
print("Test file:", TEST_FILE)
print("Maximum repairs:", MAX_RETRIES)

Real pytest runner: __main__
Solution file: /content/evaluation_cases/TC01/solution.py
Test file: /content/evaluation_cases/TC01/test_solution.py
Maximum repairs: 3


In [356]:

from pathlib import Path

cli_code = r'''
import argparse
import json
import subprocess
import sys
from pathlib import Path

BASE_DIR = Path("/content")
CASES_DIR = BASE_DIR / "evaluation_cases"


def main():
    parser = argparse.ArgumentParser(
        description="Agentic AI Software Debugging and Test Automation"
    )
    parser.add_argument(
        "--case",
        required=True,
        help="Test case ID, for example TC01, TC02, or TC03"
    )
    args = parser.parse_args()

    case_id = args.case.upper()
    case_dir = CASES_DIR / case_id
    solution = case_dir / "solution.py"
    tests = case_dir / "test_solution.py"

    if not solution.is_file() or not tests.is_file():
        print(json.dumps({
            "case_id": case_id,
            "status": "ERROR",
            "message": "Case or required files not found."
        }, indent=2))
        return 2

    # Run the existing notebook agent through a generated runner.
    runner = BASE_DIR / "_agent_cli_runner.py"
    runner.write_text(
        """
import sys
from pathlib import Path

sys.path.insert(0, str(Path('/content')))
""",
        encoding="utf-8"
    )

    print(f"Selected case: {case_id}")
    print(f"Solution: {solution}")
    print(f"Tests: {tests}")
    print()
    print("The CLI entry point is ready.")
    print("The agent integration will be connected in the next step.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

Path("/content/agent_cli.py").write_text(cli_code, encoding="utf-8")
print("CLI file created:", Path("/content/agent_cli.py").exists())


CLI file created: True


In [357]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "/content/agent_cli.py", "--case", "TC01"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

Selected case: TC01
Solution: /content/evaluation_cases/TC01/solution.py
Tests: /content/evaluation_cases/TC01/test_solution.py

The CLI entry point is ready.
The agent integration will be connected in the next step.


Exit code: 0


In [358]:
from pathlib import Path

agent_module = r'''
import ast
import json
import os
import subprocess
import sys
from pathlib import Path

from google import genai

BASE_DIR = Path("/content")
CASES_DIR = BASE_DIR / "evaluation_cases"
MAX_RETRIES = 3
MODEL = "gemini-3.5-flash-lite"


def get_client():
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError("GEMINI_API_KEY is not configured.")

    return genai.Client(api_key=key)


def run_tests(case_dir):
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "--tb=short", "test_solution.py"
        ],
        cwd=str(case_dir),
        capture_output=True,
        text=True,
        timeout=30
    )
    return result.returncode, result.stdout + result.stderr


def request_structured_repair(client, code, feedback, function_name):
    prompt = f"""
You are a Python debugging agent.

Function name: {function_name}

Current code:
{code}

Pytest feedback:
{feedback}

Return JSON with exactly these fields:
bug_type (string), explanation (string),
fixed_code (string), confidence (number from 0 to 1).

Return only valid JSON. Preserve the function name.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )

    data = json.loads(response.text)
    required = {"bug_type", "explanation", "fixed_code", "confidence"}

    if not isinstance(data, dict) or not required.issubset(data):
        raise ValueError("Invalid structured repair response.")

    if not isinstance(data["fixed_code"], str):
        raise ValueError("fixed_code must be a string.")

    if not isinstance(data["confidence"], (int, float)):
        raise ValueError("confidence must be numeric.")

    if not 0 <= data["confidence"] <= 1:
        raise ValueError("confidence must be between 0 and 1.")

    tree = ast.parse(data["fixed_code"])
    function_names = {
        node.name for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    }

    if function_name not in function_names:
        raise ValueError("Repair did not preserve the required function.")

    return data


def main():
    import argparse

    parser = argparse.ArgumentParser(
        description="Agentic AI Debugging and Test Automation"
    )
    parser.add_argument("--case", required=True, choices=["TC01", "TC02", "TC03"])
    args = parser.parse_args()

    case_dir = CASES_DIR / args.case
    solution = case_dir / "solution.py"
    test_file = case_dir / "test_solution.py"

    if not solution.is_file() or not test_file.is_file():
        print(json.dumps({
            "case_id": args.case,
            "status": "ERROR",
            "message": "Solution or test file is missing."
        }, indent=2))
        return 2

    try:
        client = get_client()
        attempts = 0

        for run_number in range(MAX_RETRIES + 1):
            code = solution.read_text(encoding="utf-8")
            exit_code, feedback = run_tests(case_dir)

            print(f"Test run {run_number + 1}: exit code {exit_code}")
            print(feedback[-1000:])

            if exit_code == 0:
                print(json.dumps({
                    "case_id": args.case,
                    "status": "PASS",
                    "repair_attempts": attempts,
                    "message": "All tests passed."
                }, indent=2))
                return 0

            if attempts >= MAX_RETRIES:
                break

            tree = ast.parse(code)
            functions = [
                node.name for node in tree.body
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            ]
            if not functions:
                raise ValueError("No function found in solution.")

            attempts += 1
            print(f"Repair attempt {attempts}/{MAX_RETRIES}")

            repair = request_structured_repair(
                client, code, feedback, functions[0]
            )

            print("Bug type:", repair["bug_type"])
            compile(repair["fixed_code"], "<candidate>", "exec")
            solution.write_text(repair["fixed_code"], encoding="utf-8")

        print(json.dumps({
            "case_id": args.case,
            "status": "FAIL",
            "repair_attempts": attempts,
            "message": "Retry limit reached."
        }, indent=2))
        return 1

    except Exception as error:
        print(json.dumps({
            "case_id": args.case,
            "status": "ERROR",
            "message": str(error)
        }, indent=2))
        return 2


if __name__ == "__main__":
    raise SystemExit(main())
'''

Path("/content/debugging_agent.py").write_text(
    agent_module, encoding="utf-8"
)
print("Standalone agent module created.")

Standalone agent module created.


In [359]:
from pathlib import Path

Path("/content/agent_cli.py").write_text(
    '''import subprocess
import sys

if __name__ == "__main__":
    result = subprocess.run([
        sys.executable,
        "/content/debugging_agent.py",
        *sys.argv[1:]
    ])
    raise SystemExit(result.returncode)
''',
    encoding="utf-8"
)

print("CLI connected to debugging agent.")

CLI connected to debugging agent.


In [360]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC01"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("Errors / warnings:")
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 0
....                                                                     [100%]
4 passed in 0.01s

{
  "case_id": "TC01",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}

CLI exit code: 0


In [361]:
from pathlib import Path

agent_path = Path("/content/debugging_agent.py")
source = agent_path.read_text(encoding="utf-8")

old_function = '''def get_client():
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError("GEMINI_API_KEY is not configured.")

    return genai.Client(api_key=key)
'''

new_function = '''def get_client():
    key = os.environ.get("GEMINI_API_KEY")

    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError(
            "GEMINI_API_KEY is unavailable. Check Colab Secrets "
            "or configure the environment variable before launching."
        )

    return genai.Client(api_key=key)
'''

if old_function not in source:
    print("The expected function was not found. No file changes made.")
else:
    agent_path.write_text(
        source.replace(old_function, new_function, 1),
        encoding="utf-8"
    )
    print("API key loading function updated.")

API key loading function updated.


In [362]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "Add GEMINI_API_KEY to Google Colab Secrets first."
    )

env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case",
        "TC01"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 0
....                                                                     [100%]
4 passed in 0.01s

{
  "case_id": "TC01",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}

CLI exit code: 0


In [363]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import userdata

case_dir = Path("/content/evaluation_cases/TC01")
solution = case_dir / "solution.py"

buggy_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

# Reset only the isolated TC01 example.
solution.write_text(buggy_code.strip() + "\n", encoding="utf-8")

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY is unavailable in Colab Secrets.")

env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC01"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print("Errors / warnings:\n", result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 1
tal / (len(numbers) - 1)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
E   ZeroDivisionError: division by zero
____________________________ test_negative_numbers _____________________________
test_solution.py:11: in test_negative_numbers
    assert calculate_average([-2, -4, -6]) == -4
E   assert -6.0 == -4
E    +  where -6.0 = calculate_average([-2, -4, -6])
_______________________________ test_empty_list ________________________________
test_solution.py:14: in test_empty_list
    with pytest.raises(ValueError):
         ^^^^^^^^^^^^^^^^^^^^^^^^^
E   Failed: DID NOT RAISE <class 'ValueError'>
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_average - assert 6.666666666666667 == 5.0
FAILED test_solution.py::test_single_element - ZeroDivisionError: division by...
FAILED test_solution.py::test_negative_numbers - assert -6.0 == -4
FAILED test_solution.py::test_empty_list - Failed: DID NOT RAISE <class 'Va

In [364]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC02"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 0
...                                                                      [100%]
3 passed in 0.01s

{
  "case_id": "TC02",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}

CLI exit code: 0


In [365]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC03"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 0
...                                                                      [100%]
3 passed in 0.01s

{
  "case_id": "TC03",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}

CLI exit code: 0


In [366]:
from pathlib import Path

tc02_solution = Path(
    "/content/evaluation_cases/TC02/solution.py"
)

buggy_code = """
def is_adult(age):
    return age > 18
"""

tc02_solution.write_text(
    buggy_code.strip() + "\n",
    encoding="utf-8"
)

print("TC02 reset to its original buggy version.")
print(tc02_solution.read_text())

TC02 reset to its original buggy version.
def is_adult(age):
    return age > 18



In [367]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY is missing from Colab Secrets.")

env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC02"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 1
.F.                                                                      [100%]
=================================== FAILURES ===================================
_______________________________ test_exactly_18 ________________________________
test_solution.py:7: in test_exactly_18
    assert is_adult(18) is True
E   assert False is True
E    +  where False = is_adult(18)
=========================== short test summary info ============================
FAILED test_solution.py::test_exactly_18 - assert False is True
1 failed, 2 passed in 0.04s

Repair attempt 1/3
Bug type: Logical Error
Test run 2: exit code 0
...                                                                      [100%]
3 passed in 0.01s

{
  "case_id": "TC02",
  "status": "PASS",
  "repair_attempts": 1,
  "message": "All tests passed."
}

CLI exit code: 0


In [368]:
from pathlib import Path

tc03_solution = Path(
    "/content/evaluation_cases/TC03/solution.py"
)

buggy_code = """
def calculate_total(prices):
    total = sum(prices)
    return tota
"""

tc03_solution.write_text(
    buggy_code.strip() + "\n",
    encoding="utf-8"
)

print("TC03 reset to its original buggy version.")
print(tc03_solution.read_text())

TC03 reset to its original buggy version.
def calculate_total(prices):
    total = sum(prices)
    return tota



In [369]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY is missing from Colab Secrets.")

env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC03"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 1
           ^^^^
E   NameError: name 'tota' is not defined
______________________________ test_single_price _______________________________
test_solution.py:7: in test_single_price
    assert calculate_total([15]) == 15
           ^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in calculate_total
    return tota
           ^^^^
E   NameError: name 'tota' is not defined
______________________________ test_empty_prices _______________________________
test_solution.py:10: in test_empty_prices
    assert calculate_total([]) == 0
           ^^^^^^^^^^^^^^^^^^^
solution.py:3: in calculate_total
    return tota
           ^^^^
E   NameError: name 'tota' is not defined
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_total - NameError: name 'tota' is not de...
FAILED test_solution.py::test_single_price - NameError: name 'tota' is not de...
FAILED test_solution.py::test_empty_prices - NameError: name 'tota' is not 

In [370]:
import csv
from pathlib import Path

results = [
    {
        "case_id": "TC01",
        "bug_type": "Off-by-one error",
        "initial_failures": 4,
        "tests_passed": 4,
        "repair_attempts": 1,
        "status": "PASS"
    },
    {
        "case_id": "TC02",
        "bug_type": "Incorrect conditional logic",
        "initial_failures": 1,
        "tests_passed": 3,
        "repair_attempts": 1,
        "status": "PASS"
    },
    {
        "case_id": "TC03",
        "bug_type": "Undefined variable",
        "initial_failures": 3,
        "tests_passed": 3,
        "repair_attempts": 1,
        "status": "PASS"
    }
]

output_file = Path("/content/week2_evaluation_results.csv")

with output_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Saved:", output_file)
print("Cases:", len(results))
print("Passed:", sum(r["status"] == "PASS" for r in results))
print("Total repair attempts:", sum(r["repair_attempts"] for r in results))

Saved: /content/week2_evaluation_results.csv
Cases: 3
Passed: 3
Total repair attempts: 3


In [371]:
from pathlib import Path

files_to_check = [
    "/content/debugging_agent.py",
    "/content/agent_cli.py",
    "/content/bug_cases.json",
    "/content/week2_evaluation_results.csv",
    "/content/evaluation_cases/TC01/test_solution.py",
    "/content/evaluation_cases/TC02/test_solution.py",
    "/content/evaluation_cases/TC03/test_solution.py",
]

for filename in files_to_check:
    path = Path(filename)
    print(
        "FOUND" if path.is_file() else "MISSING",
        filename
    )

FOUND /content/debugging_agent.py
FOUND /content/agent_cli.py
FOUND /content/bug_cases.json
FOUND /content/week2_evaluation_results.csv
FOUND /content/evaluation_cases/TC01/test_solution.py
FOUND /content/evaluation_cases/TC02/test_solution.py
FOUND /content/evaluation_cases/TC03/test_solution.py


In [372]:

from pathlib import Path

readme = r"""
# Agentic AI Software Debugging and Test Automation System

## 1. Project overview
This project demonstrates an AI debugging agent that uses Gemini to
propose Python code repairs and pytest to verify whether the repairs work.

## 2. Week 2 objectives
- Connect to the Gemini API.
- Run automated tests using pytest.
- Generate structured JSON repair responses.
- Validate generated Python syntax before saving a repair.
- Retry repairs up to a configurable limit.
- Provide a command-line interface (CLI).
- Record evaluation results in CSV format.

## 3. Architecture
1. The user selects a test case through the CLI.
2. The agent runs pytest on the selected case.
3. If tests fail, the agent sends the code and failure feedback to Gemini.
4. Gemini returns JSON containing the bug type, explanation,
   fixed code, and confidence score.
5. The agent validates the response and Python syntax.
6. The candidate is saved and pytest runs again.
7. The agent stops when tests pass or the repair limit is reached.

## 4. Project files
- `debugging_agent.py`: standalone debugging agent.
- `agent_cli.py`: command-line entry point.
- `bug_cases.json`: bug-case dataset.
- `evaluation_cases/`: isolated solutions and tests for TC01–TC03.
- `week2_evaluation_results.csv`: observed Week 2 evaluation summary.

## 5. Setup
Install the dependencies in your Python environment:

```bash
pip install google-genai pytest
```

Configure `GEMINI_API_KEY` as an environment variable before running
the standalone CLI. Do not commit real API keys to GitHub.

## 6. Run the CLI
```bash
python agent_cli.py --case TC01
python agent_cli.py --case TC02
python agent_cli.py --case TC03
```

The current implementation uses the configured case files under
`/content/evaluation_cases/`. Update `CASES_DIR` in `debugging_agent.py`
if running outside the current Colab directory structure.

## 7. Week 2 evaluation
Three controlled cases were tested:
- TC01: off-by-one error — PASS after one repair attempt.
- TC02: incorrect conditional logic — PASS after one repair attempt.
- TC03: undefined variable — PASS after one repair attempt.

Total: 3 cases passed; 3 repair attempts; 10 tests passed after repair.

These are preliminary controlled results, not a broad benchmark.

## 8. Limitations and safety
- Gemini-generated code is not guaranteed to be correct.
- Syntax validation alone does not establish correctness or safety.
- The current prototype runs generated code through pytest; do not use
  it with untrusted code without appropriate isolation.
- Test results depend on the selected tests and may miss other defects.
- Week 2 does not yet implement RAG, persistent memory, a graphical UI,
  or a complete production sandbox.

## 9. Next steps
Expand evaluation coverage, strengthen execution isolation, add logging,
and complete the remaining project lifecycle requirements.
"""

Path("/content/README.md").write_text(readme.strip() + "\n", encoding="utf-8")
print("README created:", Path("/content/README.md").is_file())
print("Location: /content/README.md")


README created: True
Location: /content/README.md


In [373]:
from pathlib import Path
import shutil

source = Path("/content")
project = source / "agentic-ai-debugging-test-automation"

project.mkdir(exist_ok=True)

# Main project files
files = [
    "debugging_agent.py",
    "agent_cli.py",
    "bug_cases.json",
    "README.md",
    "week2_evaluation_results.csv",
]

for name in files:
    src = source / name
    if src.exists():
        shutil.copy2(src, project / name)
        print("Copied:", name)
    else:
        print("MISSING:", name)

# Copy test cases and their isolated source files
cases_src = source / "evaluation_cases"
if cases_src.exists():
    shutil.copytree(
        cases_src,
        project / "evaluation_cases",
        dirs_exist_ok=True
    )
    print("Copied evaluation cases")

print("\nProject folder:", project)

Copied: debugging_agent.py
Copied: agent_cli.py
Copied: bug_cases.json
Copied: README.md
Copied: week2_evaluation_results.csv
Copied evaluation cases

Project folder: /content/agentic-ai-debugging-test-automation


In [374]:
from pathlib import Path

project = Path("/content/agentic-ai-debugging-test-automation")

gitignore = """
.env
.env.*
!.env.example
__pycache__/
*.py[cod]
.venv/
venv/
.ipynb_checkpoints/
"""

(project / ".gitignore").write_text(
    gitignore.strip() + "\n",
    encoding="utf-8"
)

print(".gitignore created")

.gitignore created


In [375]:
from pathlib import Path
import shutil

source = Path("/content")
project = source / "agentic-ai-debugging-test-automation"

project.mkdir(exist_ok=True)

files = [
    "debugging_agent.py",
    "agent_cli.py",
    "bug_cases.json",
    "README.md",
    "week2_evaluation_results.csv"
]

for name in files:
    file_path = source / name
    if file_path.exists():
        shutil.copy2(file_path, project / name)
        print("Copied:", name)
    else:
        print("Not found:", name)

cases = source / "evaluation_cases"

if cases.exists():
    shutil.copytree(cases, project / "evaluation_cases", dirs_exist_ok=True)
    print("Copied evaluation_cases folder")

print("\nYour project folder is:", project)

Copied: debugging_agent.py
Copied: agent_cli.py
Copied: bug_cases.json
Copied: README.md
Copied: week2_evaluation_results.csv
Copied evaluation_cases folder

Your project folder is: /content/agentic-ai-debugging-test-automation


In [376]:
import shutil

project = "/content/agentic-ai-debugging-test-automation"
zip_path = "/content/agentic-ai-debugging-test-automation.zip"

shutil.make_archive(
    "/content/agentic-ai-debugging-test-automation",
    "zip",
    root_dir=project
)

print("ZIP file created:", zip_path)

ZIP file created: /content/agentic-ai-debugging-test-automation.zip


In [377]:
from google.colab import files

files.download("/content/agentic-ai-debugging-test-automation.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [378]:
from pathlib import Path

project = Path("/content/agentic-ai-debugging-test-automation")

print("Project folder exists:", project.exists())

for name in [
    "debugging_agent.py",
    "agent_cli.py",
    "README.md",
    "week2_evaluation_results.csv",
]:
    path = project / name
    print(f"{name}: {'Found' if path.exists() else 'Missing'}")

print("\nEvaluation cases:")
cases = project / "evaluation_cases"

if cases.exists():
    for folder in sorted(cases.iterdir()):
        if folder.is_dir():
            print("-", folder.name)
else:
    print("Evaluation cases folder is missing")

Project folder exists: True
debugging_agent.py: Found
agent_cli.py: Found
README.md: Found
week2_evaluation_results.csv: Found

Evaluation cases:
- TC01
- TC02
- TC03
- TC04


In [379]:
%pip install -q streamlit

In [380]:
from pathlib import Path

project = Path("/content/agentic-ai-debugging-test-automation")

print("=== agent_cli.py ===")
print((project / "agent_cli.py").read_text()[:12000])

print("\n=== debugging_agent.py ===")
print((project / "debugging_agent.py").read_text()[:16000])

=== agent_cli.py ===
import subprocess
import sys

if __name__ == "__main__":
    result = subprocess.run([
        sys.executable,
        "/content/debugging_agent.py",
        *sys.argv[1:]
    ])
    raise SystemExit(result.returncode)


=== debugging_agent.py ===

import ast
import json
import os
import subprocess
import sys
from pathlib import Path

from google import genai

BASE_DIR = Path("/content")
CASES_DIR = BASE_DIR / "evaluation_cases"
MAX_RETRIES = 3
MODEL = "gemini-3.5-flash-lite"


def get_client():
    key = os.environ.get("GEMINI_API_KEY")

    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError(
            "GEMINI_API_KEY is unavailable. Check Colab Secrets "
            "or configure the environment variable before launching."
        )

    return genai.Client(api_key=key)


def run_tests(case_dir):
    r

In [381]:
from pathlib import Path

project = Path("/content/agentic-ai-debugging-test-automation")

print("=== CLI arguments and setup ===")
cli = (project / "agent_cli.py").read_text(encoding="utf-8")
print(cli[:5000])

print("\n=== Test case files ===")
for path in sorted((project / "evaluation_cases").glob("**/*")):
    if path.is_file():
        print(path.relative_to(project))

=== CLI arguments and setup ===
import subprocess
import sys

if __name__ == "__main__":
    result = subprocess.run([
        sys.executable,
        "/content/debugging_agent.py",
        *sys.argv[1:]
    ])
    raise SystemExit(result.returncode)


=== Test case files ===
evaluation_cases/TC01/.pytest_cache/.gitignore
evaluation_cases/TC01/.pytest_cache/CACHEDIR.TAG
evaluation_cases/TC01/.pytest_cache/README.md
evaluation_cases/TC01/.pytest_cache/v/cache/lastfailed
evaluation_cases/TC01/.pytest_cache/v/cache/nodeids
evaluation_cases/TC01/__pycache__/solution.cpython-313.pyc
evaluation_cases/TC01/__pycache__/test_solution.cpython-313-pytest-8.4.2.pyc
evaluation_cases/TC01/solution.py
evaluation_cases/TC01/test_solution.py
evaluation_cases/TC02/.pytest_cache/.gitignore
evaluation_cases/TC02/.pytest_cache/CACHEDIR.TAG
evaluation_cases/TC02/.pytest_cache/README.md
evaluation_cases/TC02/.pytest_cache/v/cache/lastfailed
evaluation_cases/TC02/.pytest_cache/v/cache/nodeids
evaluation_cases

In [382]:
from pathlib import Path

project = Path("/content/agentic-ai-debugging-test-automation")

app_code = r'''
import streamlit as st
import subprocess
import sys
import os
import difflib
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

st.set_page_config(
    page_title="Agentic AI Debugging System",
    page_icon="🐞",
    layout="wide"
)

st.title("🐞 Agentic AI Debugging & Test Automation")
st.caption("AI-assisted bug detection, code repair, and automated test validation")

st.info(
    "Select a test case and run the existing debugging agent. "
    "The agent may modify the selected case's solution file while repairing it."
)

case_id = st.selectbox(
    "Select a test case",
    ["TC01", "TC02", "TC03"]
)

case_dir = PROJECT / "evaluation_cases" / case_id
solution_file = case_dir / "solution.py"

if solution_file.exists():
    original_code = solution_file.read_text(encoding="utf-8")
    with st.expander("View current solution code"):
        st.code(original_code, language="python")
else:
    original_code = ""
    st.error(f"Solution file not found: {solution_file}")

if st.button("🚀 Run AI Debugging Agent", type="primary", disabled=not solution_file.exists()):
    if not os.environ.get("GEMINI_API_KEY"):
        st.error(
            "GEMINI_API_KEY is not available to the app. "
            "Set it securely in the Colab launcher cell, then restart the app."
        )
        st.stop()

    with st.spinner("Running the debugging agent and automated tests..."):
        try:
            result = subprocess.run(
                [
                    sys.executable,
                    str(PROJECT / "agent_cli.py"),
                    "--case",
                    case_id
                ],
                cwd=str(PROJECT),
                capture_output=True,
                text=True,
                timeout=240,
                env=os.environ.copy()
            )

            output = result.stdout or ""
            errors = result.stderr or ""
            repaired_code = solution_file.read_text(encoding="utf-8") if solution_file.exists() else ""

            st.subheader("1. Agent execution result")
            if result.returncode == 0:
                st.success("Agent finished successfully. Check the test output below.")
            elif result.returncode == 1:
                st.warning("The agent finished, but some tests may still be failing.")
            else:
                st.error("The agent encountered an error. Review the output below.")

            st.subheader("2. Agent log and test results")
            st.code(output[-12000:] if output else "No standard output was returned.")

            if errors.strip():
                with st.expander("View error details"):
                    st.code(errors[-6000:])

            st.subheader("3. Repaired solution code")
            st.code(repaired_code, language="python")

            st.subheader("4. Code changes")
            diff = difflib.unified_diff(
                original_code.splitlines(),
                repaired_code.splitlines(),
                fromfile="Before repair",
                tofile="After repair",
                lineterm=""
            )
            diff_text = "\n".join(diff)
            if diff_text:
                st.code(diff_text, language="diff")
            else:
                st.info("No code changes were detected.")

            st.caption(
                "The agent log contains the repair attempt and final status. "
                "A successful run should report PASS and all tests passing."
            )

        except subprocess.TimeoutExpired:
            st.error("The agent took too long and was stopped after 4 minutes.")
        except Exception as exc:
            st.error(f"Could not run the agent: {exc}")
'''

app_path = project / "app.py"
app_path.write_text(app_code, encoding="utf-8")

print("Streamlit app created:", app_path)
print("Existing debugging agent files were not edited.")

Streamlit app created: /content/agentic-ai-debugging-test-automation/app.py
Existing debugging agent files were not edited.


In [383]:
import os
import subprocess
import sys
from google.colab import userdata

# Get the API key from Colab Secrets without displaying it
api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it under Colab Secrets and try again."
    )

# Pass the key to the app process through its environment
app_env = os.environ.copy()
app_env["GEMINI_API_KEY"] = api_key

# Start Streamlit
app_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        "/content/agentic-ai-debugging-test-automation/app.py",
        "--server.port", "8501",
        "--server.headless", "true"
    ],
    env=app_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Streamlit launch requested.")
print("App process ID:", app_process.pid)
print("Your API key has not been printed.")

Streamlit launch requested.
App process ID: 5456
Your API key has not been printed.


In [384]:
from google.colab import output

print("Opening your Agentic AI Debugging UI...")
display(output.serve_kernel_port_as_window(8501))

Opening your Agentic AI Debugging UI...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

None

In [385]:
import urllib.request
import subprocess
import time
from pathlib import Path

try:
    response = urllib.request.urlopen(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=5
    )
    print("Streamlit health check:", response.read().decode())
except Exception as e:
    print("Streamlit health check failed:", str(e))

print("\nChecking app process...")
if "app_process" in globals():
    print("Process running:", app_process.poll() is None)
else:
    print("Previous process variable not found.")

print("\nChecking app file...")
print("app.py exists:", Path(
    "/content/agentic-ai-debugging-test-automation/app.py"
).exists())

Streamlit health check: ok

Checking app process...
Process running: True

Checking app file...
app.py exists: True


In [386]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

In [387]:
import urllib.request

for path in ["/", "/_stcore/health"]:
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8501" + path,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req, timeout=10) as response:
            print(path, "Status:", response.status)
            print(response.read(500).decode("utf-8", errors="replace"))
    except Exception as e:
        print(path, "Error:", e)

/ Status: 200
<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or im
/_stcore/health Status: 200
ok


In [388]:
import subprocess

result = subprocess.run(
    ["bash", "-lc", "ps -ef | grep '[s]treamlit run'"],
    capture_output=True,
    text=True
)
print(result.stdout or "No Streamlit process found.")

root        2137    1308  1 10:43 ?        00:00:09 /usr/bin/python3 -m streamlit run /content/agentic-ai-debugging-test-automation/app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false --browser.gatherUsageStats false
root        5456    1308 55 10:54 ?        00:00:00 /usr/bin/python3 -m streamlit run /content/agentic-ai-debugging-test-automation/app.py --server.port 8501 --server.headless true



In [389]:
app_process.terminate()
app_process.wait(timeout=10)
print("Previous Streamlit process stopped.")

Previous Streamlit process stopped.


In [390]:
import os
import subprocess
import sys
from google.colab import userdata

project = "/content/agentic-ai-debugging-test-automation"

app_env = os.environ.copy()
app_env["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

app_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        f"{project}/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--logger.level", "info"
    ],
    env=app_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Streamlit restarted. Process ID:", app_process.pid)

Streamlit restarted. Process ID: 5496


In [391]:
import time

time.sleep(3)

if app_process.poll() is None:
    print("Streamlit is running.")
    print("Recent startup logs:")
    for _ in range(20):
        import os
        import select
        ready, _, _ = select.select([app_process.stdout], [], [], 0.2)
        if not ready:
            break
        line = app_process.stdout.readline()
        if not line:
            break
        print(line.rstrip())
else:
    print("Streamlit exited with code:", app_process.returncode)
    print(app_process.stdout.read())

Streamlit is running.
Recent startup logs:



In [392]:
import urllib.request

try:
    with urllib.request.urlopen(
        "http://127.0.0.1:8501/",
        timeout=10
    ) as response:
        html = response.read().decode("utf-8", errors="replace")
        print("HTTP status:", response.status)
        print("Streamlit page received:", "streamlit" in html.lower())
        print("Page size:", len(html), "characters")
except Exception as e:
    print("Connection error:", e)

HTTP status: 200
Streamlit page received: True
Page size: 6467 characters


In [393]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

In [394]:
output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

In [395]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(8501)")
print(url)

https://8501-m-s-use1b10-3ncc0aujta5en-b.us-east1.prod.r.colab.dev


In [396]:
import subprocess

result = subprocess.run(
    [
        "bash", "-lc",
        "curl -s http://127.0.0.1:8501/_stcore/health; "
        "echo; "
        "curl -s http://127.0.0.1:8501/_stcore/host-config"
    ],
    capture_output=True,
    text=True,
    timeout=15
)

print(result.stdout)
print("Errors:", result.stderr)

ok
{"allowedOrigins":["https://devel.streamlit.test","https://*.streamlit.apptest","https://*.streamlitapp.test","https://*.streamlitapp.com","https://share.streamlit.io","https://share-demo.streamlit.io","https://share-head.streamlit.io","https://share-staging.streamlit.io","https://*.demo.streamlit.run","https://*.head.streamlit.run","https://*.staging.streamlit.run","https://*.streamlit.run","https://*.demo.streamlit.app","https://*.head.streamlit.app","https://*.staging.streamlit.app","https://*.streamlit.app"],"useExternalAuthToken":false,"enableCustomParentMessages":false,"enforceDownloadInNewTab":false,"metricsUrl":"","blockErrorDialogs":false,"resourceCrossOriginMode":null}
Errors: 


In [397]:
import os
import signal

if app_process.poll() is None:
    print("Streamlit process is still running.")
    print("PID:", app_process.pid)
    print("Log output is currently being captured by the running process.")
else:
    print("Streamlit exited with code:", app_process.returncode)

Streamlit exited with code: 1


In [398]:
import os
import subprocess
import sys
import time
from google.colab import userdata

# Stop the current Streamlit process
if app_process.poll() is None:
    app_process.terminate()
    app_process.wait(timeout=10)

project = "/content/agentic-ai-debugging-test-automation"
log_path = "/content/streamlit_debug.log"

env = os.environ.copy()
env["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

with open(log_path, "w") as log_file:
    app_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run",
            f"{project}/app.py",
            "--server.port", "8501",
            "--server.headless", "true",
            "--logger.level", "debug"
        ],
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

time.sleep(3)

print("Streamlit process running:", app_process.poll() is None)
print("Log file created:", os.path.exists(log_path))
print("\nRecent logs:\n")
print(open(log_path, encoding="utf-8").read()[-6000:])

Streamlit process running: True
Log file created: True

Recent logs:



2026-11-02 10:54:43.699 No ASGI app assignments found in /content/agentic-ai-debugging-test-automation/app.py



In [399]:
from google.colab import output

display(output.serve_kernel_port_as_iframe(8501, height=800))

<IPython.core.display.Javascript object>

None

In [400]:
from pathlib import Path

log = Path("/content/streamlit_debug.log").read_text(
    encoding="utf-8", errors="replace"
)

print(log[-10000:])



2026-11-02 10:54:43.699 No ASGI app assignments found in /content/agentic-ai-debugging-test-automation/app.py



In [401]:
import os
import subprocess
import sys
import time
from google.colab import userdata

# Stop the existing Streamlit process
if app_process.poll() is None:
    app_process.terminate()
    app_process.wait(timeout=10)

project = "/content/agentic-ai-debugging-test-automation"
env = os.environ.copy()
env["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

app_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        f"{project}/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false"
    ],
    env=env,
    stdout=open("/content/streamlit_debug.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(3)
print("Streamlit running:", app_process.poll() is None)

Streamlit running: True


In [402]:
from google.colab import output

display(output.serve_kernel_port_as_iframe(8501, height=800))

<IPython.core.display.Javascript object>

None

In [403]:
from pathlib import Path

project = Path("/content/agentic-ai-debugging-test-automation")

for case_id in ["TC01", "TC02", "TC03"]:
    solution = project / "evaluation_cases" / case_id / "solution.py"

    print(f"\n{'=' * 12} {case_id} {'=' * 12}")
    print(solution.read_text(encoding="utf-8"))


============ TC01 ============
def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / len(numbers)

============ TC02 ============
def is_adult(age):
    return age >= 18

============ TC03 ============
def calculate_total(prices):
    total = sum(prices)
    return total


In [404]:
from pathlib import Path
import shutil

project = Path("/content/agentic-ai-debugging-test-automation")
backup = project / "repaired_solutions_backup"
backup.mkdir(exist_ok=True)

for case_id in ["TC01", "TC02", "TC03"]:
    source = project / "evaluation_cases" / case_id / "solution.py"
    destination = backup / f"{case_id}_solution.py"
    shutil.copy2(source, destination)
    print(f"{case_id}: working solution backed up")

print("\nAll three working solutions are safely backed up.")

TC01: working solution backed up
TC02: working solution backed up
TC03: working solution backed up

All three working solutions are safely backed up.


In [405]:
from pathlib import Path

project = Path("/content/agentic-ai-debugging-test-automation")

buggy_solutions = {
    "TC01": '''def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / (len(numbers) - 1)
''',
    "TC02": '''def is_adult(age):
    return age > 18
''',
    "TC03": '''def calculate_total(prices):
    total = sum(prices)
    return tota
'''
}

for case_id, code in buggy_solutions.items():
    solution_file = project / "evaluation_cases" / case_id / "solution.py"
    solution_file.write_text(code, encoding="utf-8")
    print(f"{case_id}: original buggy code restored")

print("\nReady to demonstrate AI debugging.")

TC01: original buggy code restored
TC02: original buggy code restored
TC03: original buggy code restored

Ready to demonstrate AI debugging.


In [406]:
from pathlib import Path

print("=== Main agent file ===")
agent = Path("/content/debugging_agent.py")
print("Exists:", agent.exists())

if agent.exists():
    for line_no, line in enumerate(
        agent.read_text(encoding="utf-8").splitlines(), 1
    ):
        if "case_dir" in line or "evaluation_cases" in line or "solution" in line:
            print(f"{line_no}: {line}")

print("\n=== TC01 files ===")
for path in [
    Path("/content/evaluation_cases/TC01/solution.py"),
    Path("/content/agentic-ai-debugging-test-automation/evaluation_cases/TC01/solution.py"),
]:
    print(f"\n{path}")
    print("Exists:", path.exists())
    if path.exists():
        print(path.read_text(encoding="utf-8"))

=== Main agent file ===
Exists: True
12: CASES_DIR = BASE_DIR / "evaluation_cases"
36: def run_tests(case_dir):
40:             "-q", "--tb=short", "test_solution.py"
42:         cwd=str(case_dir),
111:     case_dir = CASES_DIR / args.case
112:     solution = case_dir / "solution.py"
113:     test_file = case_dir / "test_solution.py"
115:     if not solution.is_file() or not test_file.is_file():
128:             code = solution.read_text(encoding="utf-8")
129:             exit_code, feedback = run_tests(case_dir)
152:                 raise ValueError("No function found in solution.")
163:             solution.write_text(repair["fixed_code"], encoding="utf-8")

=== TC01 files ===

/content/evaluation_cases/TC01/solution.py
Exists: True
def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / len(numbers)

/content/agentic-ai-debugging-test-automation/evaluation_cases/TC01/solution.py
Exists: True
def

In [407]:
from pathlib import Path

agent_file = Path("/content/debugging_agent.py")
code = agent_file.read_text(encoding="utf-8")

old = 'CASES_DIR = BASE_DIR / "evaluation_cases"'
new = 'CASES_DIR = Path("/content/agentic-ai-debugging-test-automation/evaluation_cases")'

if old in code:
    code = code.replace(old, new, 1)
    agent_file.write_text(code, encoding="utf-8")
    print("Fixed: debugging agent now uses the project evaluation_cases folder.")
elif new in code:
    print("The agent path is already fixed.")
else:
    print("Could not find the expected line. No changes made.")
    print("Please send me the first 25 lines of debugging_agent.py.")

Fixed: debugging agent now uses the project evaluation_cases folder.


In [408]:
from pathlib import Path

agent_file = Path("/content/debugging_agent.py")
project = Path("/content/agentic-ai-debugging-test-automation")

for case_id in ["TC01", "TC02", "TC03"]:
    solution = project / "evaluation_cases" / case_id / "solution.py"
    print(f"\n{case_id}:")
    print(solution.read_text(encoding="utf-8"))

print("\nAgent configured to use project folder:")
for line in agent_file.read_text(encoding="utf-8").splitlines():
    if "CASES_DIR =" in line:
        print(line)


TC01:
def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / (len(numbers) - 1)


TC02:
def is_adult(age):
    return age > 18


TC03:
def calculate_total(prices):
    total = sum(prices)
    return tota


Agent configured to use project folder:
CASES_DIR = Path("/content/agentic-ai-debugging-test-automation/evaluation_cases")


In [409]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-use1b10-3ncc0aujta5en-b.us-east1.prod.r.colab.dev


In [410]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [411]:
!git clone https://github.com/shobhis330/agentic-ai-debugging-test-automation.git

fatal: destination path 'agentic-ai-debugging-test-automation' already exists and is not an empty directory.


In [412]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")

os.chdir(PROJECT_DIR)

print("Project directory:", PROJECT_DIR)
print("\nProject files:")

for file in PROJECT_DIR.iterdir():
    print(file.name)

Project directory: /content/agentic-ai-debugging-test-automation

Project files:
.gitignore
test_debugger.py
evaluation_cases
debugging_agent_backup.py
README.md
week2_evaluation_results.csv
repaired_solutions_backup
agentic-ai-debugging-test-automation
__pycache__
solution.py
debugging_agent.py
.pytest_cache
agent_cli.py
app.py
test_average.py
bug_cases.json


In [413]:
from pathlib import Path

cli_path = PROJECT_DIR / "agent_cli.py"

cli_code = cli_path.read_text()

cli_code = cli_code.replace(
    '"/content/debugging_agent.py"',
    'str(Path(__file__).resolve().parent / "debugging_agent.py")'
)

if "from pathlib import Path" not in cli_code:
    cli_code = "from pathlib import Path\n" + cli_code

cli_path.write_text(cli_code)

print("Updated agent_cli.py")

Updated agent_cli.py


In [414]:
!python agent_cli.py --help

usage: debugging_agent.py [-h] --case {TC01,TC02,TC03}

Agentic AI Debugging and Test Automation

options:
  -h, --help            show this help message and exit
  --case {TC01,TC02,TC03}


In [415]:
%cd /content/agentic-ai-debugging-test-automation

!python agent_cli.py --case TC01

/content/agentic-ai-debugging-test-automation
Test run 1: exit code 0
....                                                                     [100%]
4 passed in 0.01s

{
  "case_id": "TC01",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}


In [416]:
from google.colab import userdata
import os

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("Gemini API key loaded successfully!")
except Exception as e:
    print("API key not available. Check Colab Secrets and notebook access.")

Gemini API key loaded successfully!


In [417]:
import os

print(
    "API key available:",
    bool(os.environ.get("GEMINI_API_KEY"))
)

API key available: True


In [418]:
%cd /content/agentic-ai-debugging-test-automation

!python agent_cli.py --case TC01

/content/agentic-ai-debugging-test-automation
Test run 1: exit code 0
....                                                                     [100%]
4 passed in 0.01s

{
  "case_id": "TC01",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}


In [419]:
from pathlib import Path

case_dir = Path("evaluation_cases/TC01")

for file in case_dir.iterdir():
    if file.is_file():
        print(f"\n{'='*50}")
        print("FILE:", file.name)
        print('='*50)
        print(file.read_text())


FILE: test_solution.py
import pytest
from solution import calculate_average

def test_normal_average():
    assert calculate_average([2, 4, 6, 8]) == 5.0

def test_single_element():
    assert calculate_average([10]) == 10

def test_negative_numbers():
    assert calculate_average([-2, -4, -6]) == -4

def test_empty_list():
    with pytest.raises(ValueError):
        calculate_average([])


FILE: solution.py
def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [420]:
from pathlib import Path

solution = Path("evaluation_cases/TC01/solution.py")

print(solution.read_text())

def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [421]:
!python agent_cli.py --case TC01

Test run 1: exit code 0
....                                                                     [100%]
4 passed in 0.01s

{
  "case_id": "TC01",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}


In [422]:
import os
from pathlib import Path

print("Current directory:", os.getcwd())

print("\nTC01 solution path:")
print(Path("evaluation_cases/TC01/solution.py").resolve())

print("\nBuggy formula present:")
print("(len(numbers) - 1)" in Path(
    "evaluation_cases/TC01/solution.py"
).read_text())

Current directory: /content/agentic-ai-debugging-test-automation

TC01 solution path:
/content/agentic-ai-debugging-test-automation/evaluation_cases/TC01/solution.py

Buggy formula present:
True


In [423]:
%cd /content/agentic-ai-debugging-test-automation/evaluation_cases/TC01

!python -m pytest -q test_solution.py

/content/agentic-ai-debugging-test-automation/evaluation_cases/TC01
FFF.                                                                     [100%]
=================================== FAILURES ===================================
_____________________________ test_normal_average ______________________________

    def test_normal_average():
>       assert calculate_average([2, 4, 6, 8]) == 5.0
E       assert 6.666666666666667 == 5.0
E        +  where 6.666666666666667 = calculate_average([2, 4, 6, 8])

/content/evaluation_cases/TC01/test_solution.py:5: AssertionError
_____________________________ test_single_element ______________________________

    def test_single_element():
>       assert calculate_average([10]) == 10
               ^^^^^^^^^^^^^^^^^^^^^^^

/content/evaluation_cases/TC01/test_solution.py:8: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

numbers = [10]

    def calculate_average(numbers):
        if not numbers:
            raise V

In [424]:
from pathlib import Path

agent_file = Path("/content/agentic-ai-debugging-test-automation/debugging_agent.py")

print(agent_file.read_text())



import ast
import json
import os
import subprocess
import sys
from pathlib import Path

from google import genai

BASE_DIR = Path("/content")
CASES_DIR = BASE_DIR / "evaluation_cases"
MAX_RETRIES = 3
MODEL = "gemini-3.5-flash-lite"


def get_client():
    key = os.environ.get("GEMINI_API_KEY")

    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError(
            "GEMINI_API_KEY is unavailable. Check Colab Secrets "
            "or configure the environment variable before launching."
        )

    return genai.Client(api_key=key)


def run_tests(case_dir):
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "--tb=short", "test_solution.py"
        ],
        cwd=str(case_dir),
        capture_output=True,
        text=True,
        timeout=30
    )
    return result.returncode,

In [425]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")

agent_file = PROJECT_DIR / "debugging_agent.py"

code = agent_file.read_text(encoding="utf-8")

old = '''BASE_DIR = Path("/content")
CASES_DIR = BASE_DIR / "evaluation_cases"'''

new = '''BASE_DIR = Path(__file__).resolve().parent
CASES_DIR = BASE_DIR / "evaluation_cases"'''

if old in code:
    code = code.replace(old, new)
    agent_file.write_text(code, encoding="utf-8")
    print("SUCCESS: Project paths corrected!")
else:
    print("Path configuration not found. Check the current file.")

SUCCESS: Project paths corrected!


In [426]:
import sys

sys.path.insert(0, str(PROJECT_DIR))

import debugging_agent

print("Project directory:", debugging_agent.BASE_DIR)
print("Cases directory:", debugging_agent.CASES_DIR)

for case in ["TC01", "TC02", "TC03"]:
    print(
        case,
        "Exists:",
        (debugging_agent.CASES_DIR / case / "solution.py").exists()
    )

Project directory: /content/agentic-ai-debugging-test-automation
Cases directory: /content/agentic-ai-debugging-test-automation/evaluation_cases
TC01 Exists: True
TC02 Exists: True
TC03 Exists: True


In [427]:
solution = PROJECT_DIR / "evaluation_cases/TC01/solution.py"

print(solution.read_text())

def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [428]:
%cd /content/agentic-ai-debugging-test-automation

!python agent_cli.py --case TC01

/content/agentic-ai-debugging-test-automation
Test run 1: exit code 1
____________________________ test_single_element ______________________________
/content/evaluation_cases/TC01/test_solution.py:8: in test_single_element
    assert calculate_average([10]) == 10
           ^^^^^^^^^^^^^^^^^^^^^^^
solution.py:5: in calculate_average
    return total / (len(numbers) - 1)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
E   ZeroDivisionError: division by zero
____________________________ test_negative_numbers _____________________________
/content/evaluation_cases/TC01/test_solution.py:11: in test_negative_numbers
    assert calculate_average([-2, -4, -6]) == -4
E   assert -6.0 == -4
E    +  where -6.0 = calculate_average([-2, -4, -6])
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_average - assert 6.666666666666667 == 5.0
FAILED test_solution.py::test_single_element - ZeroDivisionError: division by...
FAILED test_solution.py::

In [429]:
%cd /content/agentic-ai-debugging-test-automation

!python agent_cli.py --case TC02

/content/agentic-ai-debugging-test-automation
Test run 1: exit code 1
.F.                                                                      [100%]
=================================== FAILURES ===================================
_______________________________ test_exactly_18 ________________________________
/content/evaluation_cases/TC02/test_solution.py:7: in test_exactly_18
    assert is_adult(18) is True
E   assert False is True
E    +  where False = is_adult(18)
=========================== short test summary info ============================
FAILED test_solution.py::test_exactly_18 - assert False is True
1 failed, 2 passed in 0.04s

Repair attempt 1/3
Bug type: off-by-one error
Test run 2: exit code 0
...                                                                      [100%]
3 passed in 0.01s

{
  "case_id": "TC02",
  "status": "PASS",
  "repair_attempts": 1,
  "message": "All tests passed."
}


In [430]:
%cd /content/agentic-ai-debugging-test-automation

!python agent_cli.py --case TC03

/content/agentic-ai-debugging-test-automation
Test run 1: exit code 1
__________________________ test_single_price _______________________________
/content/evaluation_cases/TC03/test_solution.py:7: in test_single_price
    assert calculate_total([15]) == 15
           ^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in calculate_total
    return tota
           ^^^^
E   NameError: name 'tota' is not defined
______________________________ test_empty_prices _______________________________
/content/evaluation_cases/TC03/test_solution.py:10: in test_empty_prices
    assert calculate_total([]) == 0
           ^^^^^^^^^^^^^^^^^^^
solution.py:3: in calculate_total
    return tota
           ^^^^
E   NameError: name 'tota' is not defined
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_total - NameError: name 'tota' is not de...
FAILED test_solution.py::test_single_price - NameError: name 'tota' is not de...
FAILED test_solution.py::test

In [431]:
import pandas as pd
from pathlib import Path

csv_path = Path("week2_evaluation_results.csv")

if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df)
else:
    print("Evaluation CSV not found!")

,case_id,bug_type,initial_failures,tests_passed,repair_attempts,status
0,TC01,Off-by-one error,4,4,1,PASS
1,TC02,Incorrect conditional logic,1,3,1,PASS
2,TC03,Undefined variable,3,3,1,PASS


In [432]:
import pandas as pd

data = [
    {
        "case_id": "TC01",
        "bug_type": "Off-by-one error",
        "initial_failures": 3,
        "tests_passed": 4,
        "repair_attempts": 1,
        "status": "PASS"
    },
    {
        "case_id": "TC02",
        "bug_type": "Incorrect conditional logic",
        "initial_failures": 1,
        "tests_passed": 3,
        "repair_attempts": 1,
        "status": "PASS"
    },
    {
        "case_id": "TC03",
        "bug_type": "Undefined variable",
        "initial_failures": 3,
        "tests_passed": 3,
        "repair_attempts": 1,
        "status": "PASS"
    }
]

df = pd.DataFrame(data)

df.to_csv("week2_evaluation_results.csv", index=False)

display(df)
print("Evaluation CSV updated successfully!")

,case_id,bug_type,initial_failures,tests_passed,repair_attempts,status
0,TC01,Off-by-one error,3,4,1,PASS
1,TC02,Incorrect conditional logic,1,3,1,PASS
2,TC03,Undefined variable,3,3,1,PASS


Evaluation CSV updated successfully!


In [433]:
from pathlib import Path

agent_file = Path("debugging_agent.py")
code = agent_file.read_text()

start = code.find("for run_number")
end = code.find('if __name__ == "__main__":')

print(code[start:end])

for run_number in range(MAX_RETRIES + 1):
            code = solution.read_text(encoding="utf-8")
            exit_code, feedback = run_tests(case_dir)

            print(f"Test run {run_number + 1}: exit code {exit_code}")
            print(feedback[-1000:])

            if exit_code == 0:
                print(json.dumps({
                    "case_id": args.case,
                    "status": "PASS",
                    "repair_attempts": attempts,
                    "message": "All tests passed."
                }, indent=2))
                return 0

            if attempts >= MAX_RETRIES:
                break

            tree = ast.parse(code)
            functions = [
                node.name for node in tree.body
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            ]
            if not functions:
                raise ValueError("No function found in solution.")

            attempts += 1
            print(f"Repair attempt {attempts}/{MAX

In [434]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")

agent_file = PROJECT_DIR / "debugging_agent.py"
code = agent_file.read_text(encoding="utf-8")

old = 'parser.add_argument("--case", required=True, choices=["TC01", "TC02", "TC03"])'

new = 'parser.add_argument("--case", required=True, help="Test case ID, e.g. TC01")'

if old in code:
    code = code.replace(old, new)
    agent_file.write_text(code, encoding="utf-8")
    print("SUCCESS: Agent now accepts dynamic test case IDs.")
else:
    print("Argument configuration not found. Please share that section.")

SUCCESS: Agent now accepts dynamic test case IDs.


In [435]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC04"
CASE_DIR.mkdir(parents=True, exist_ok=True)

# Intentionally buggy code: missing colon
(CASE_DIR / "solution.py").write_text("""
def calculate_square(number)
    return number * number
""")

# Expected correct behaviour
(CASE_DIR / "test_solution.py").write_text("""
from solution import calculate_square

def test_square_positive():
    assert calculate_square(4) == 16

def test_square_zero():
    assert calculate_square(0) == 0

def test_square_negative():
    assert calculate_square(-3) == 9
""")

print("TC04 created successfully.")

TC04 created successfully.


In [436]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")

print(PROJECT_DIR / "debugging_agent.py")

/content/agentic-ai-debugging-test-automation/debugging_agent.py


In [437]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC04

/content/agentic-ai-debugging-test-automation
Test run 1: exit code 2
 in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<frozen importlib._bootstrap>:1404: in _gcd_import
    ???
<frozen importlib._bootstrap>:1369: in _find_and_load
    ???
<frozen importlib._bootstrap>:1340: in _find_and_load_unlocked
    ???
<frozen importlib._bootstrap>:935: in _load_unlocked
    ???
/usr/local/lib/python3.13/dist-packages/_pytest/assertion/rewrite.py:186: in exec_module
    exec(co, module.__dict__)
test_solution.py:2: in <module>
    from solution import calculate_square
E     File "/content/agentic-ai-debugging-test-automation/evaluation_cases/TC04/solution.py", line 2
E       def calculate_square(number)
E                                   ^
E   SyntaxError: expected ':'
=========================== short test summary info ============================
ERROR test_solution.py
!!!!!!!!!!!!!!!!!!!! Interru

In [438]:
def get_function_name(code, test_file):

    try:
        tree = ast.parse(code)

        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                return node.name

    except SyntaxError:
        print("Syntax error detected. Identifying function from tests...")

    test_code = test_file.read_text(encoding="utf-8")
    test_tree = ast.parse(test_code)

    for node in ast.walk(test_tree):
        if isinstance(node, ast.ImportFrom) and node.module == "solution":
            if node.names:
                return node.names[0].name

    raise ValueError("Could not identify the required function.")

In [439]:
print("Data type:", type(data))
print("Data value:", data)

Data type: <class 'list'>
Data value: [{'case_id': 'TC01', 'bug_type': 'Off-by-one error', 'initial_failures': 3, 'tests_passed': 4, 'repair_attempts': 1, 'status': 'PASS'}, {'case_id': 'TC02', 'bug_type': 'Incorrect conditional logic', 'initial_failures': 1, 'tests_passed': 3, 'repair_attempts': 1, 'status': 'PASS'}, {'case_id': 'TC03', 'bug_type': 'Undefined variable', 'initial_failures': 3, 'tests_passed': 3, 'repair_attempts': 1, 'status': 'PASS'}]


In [440]:

from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
AGENT_FILE = PROJECT_DIR / "debugging_agent.py"

# Keep a backup of your existing agent
if AGENT_FILE.exists():
    backup_file = PROJECT_DIR / "debugging_agent_backup.py"
    backup_file.write_text(
        AGENT_FILE.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

complete_code = r'''
import ast
import json
import os
import re
import subprocess
import sys
from pathlib import Path

from google import genai

# Configuration
BASE_DIR = Path(__file__).resolve().parent
CASES_DIR = BASE_DIR / "evaluation_cases"
MAX_RETRIES = 3
MODEL = "gemini-3.5-flash-lite"


# Gemini client
def get_client():
    key = os.environ.get("GEMINI_API_KEY")

    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError("GEMINI_API_KEY is unavailable. Check Colab Secrets.")

    return genai.Client(api_key=key)


# Run pytest
def run_tests(case_dir):
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "--tb=short", "test_solution.py"
        ],
        cwd=str(case_dir),
        capture_output=True,
        text=True,
        timeout=30
    )
    return result.returncode, result.stdout + result.stderr


# Identify function name
def get_function_name(code, test_file):
    try:
        tree = ast.parse(code)

        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                return node.name

    except SyntaxError:
        print("Syntax error detected. Checking test file.")

    test_code = test_file.read_text(encoding="utf-8")
    test_tree = ast.parse(test_code)

    for node in ast.walk(test_tree):
        if isinstance(node, ast.ImportFrom) and node.module == "solution":
            if node.names:
                return node.names[0].name

    raise ValueError("Could not identify required function.")


# Ask Gemini to repair code
def request_structured_repair(client, code, feedback, function_name):
    prompt = f"""
You are an expert Python debugging agent.

Required function name: {function_name}

Original Python code:
{code}

Pytest feedback:
{feedback}

Instructions:
1. Identify the bug.
2. Explain the error.
3. Correct the code.
4. Preserve the required function name.
5. Follow the expected behaviour in the tests.
6. Return complete corrected Python code.

Return only a JSON object with exactly these fields:
{{
    "bug_type": "string",
    "explanation": "string",
    "fixed_code": "complete corrected Python code",
    "confidence": 0.95
}}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )

    repair_data = json.loads(response.text)

    if isinstance(repair_data, list):
        if len(repair_data) == 1:
            repair_data = repair_data[0]
        else:
            raise ValueError("Unexpected Gemini response list.")

    if not isinstance(repair_data, dict):
        raise ValueError("Gemini response must be a JSON object.")

    required = {"bug_type", "explanation", "fixed_code", "confidence"}

    if not required.issubset(repair_data):
        raise ValueError("Required JSON fields are missing.")

    if not isinstance(repair_data["fixed_code"], str):
        raise ValueError("fixed_code must be a string.")

    confidence = repair_data["confidence"]

    if not isinstance(confidence, (int, float)) or not 0 <= confidence <= 1:
        raise ValueError("Invalid confidence value.")

    # Validate candidate code before saving
    tree = ast.parse(repair_data["fixed_code"])

    function_names = {
        node.name for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    }

    if function_name not in function_names:
        raise ValueError("AI repair did not preserve the required function.")

    compile(repair_data["fixed_code"], "<candidate>", "exec")

    return repair_data


# Main agent
def main():
    import argparse

    parser = argparse.ArgumentParser(
        description="Agentic AI Debugging and Test Automation"
    )
    parser.add_argument("--case", required=True, help="Example: TC01 or TC04")
    args = parser.parse_args()

    if not re.fullmatch(r"TC\d{2}", args.case):
        print(json.dumps({
            "case_id": args.case,
            "status": "ERROR",
            "message": "Invalid test case ID."
        }, indent=2))
        return 2

    case_dir = CASES_DIR / args.case
    solution = case_dir / "solution.py"
    test_file = case_dir / "test_solution.py"

    if not solution.is_file() or not test_file.is_file():
        print(json.dumps({
            "case_id": args.case,
            "status": "ERROR",
            "message": "Solution or test file is missing."
        }, indent=2))
        return 2

    try:
        client = get_client()
        attempts = 0

        print("\n================================")
        print("AGENTIC AI DEBUGGING SYSTEM")
        print("================================")
        print("Test Case:", args.case)
        print("Maximum Repairs:", MAX_RETRIES)

        for run_number in range(MAX_RETRIES + 1):
            code = solution.read_text(encoding="utf-8")

            print(f"\nTest Run {run_number + 1}")

            exit_code, feedback = run_tests(case_dir)

            print("Exit Code:", exit_code)
            print(feedback[-1500:])

            if exit_code == 0:
                print("\nALL TESTS PASSED!")
                print(json.dumps({
                    "case_id": args.case,
                    "status": "PASS",
                    "repair_attempts": attempts,
                    "message": "All tests passed."
                }, indent=2))
                return 0

            if attempts >= MAX_RETRIES:
                break

            function_name = get_function_name(code, test_file)

            attempts += 1
            print(f"\nRepair Attempt {attempts}/{MAX_RETRIES}")

            repair = request_structured_repair(
                client, code, feedback, function_name
            )

            print("Bug Type:", repair["bug_type"])
            print("Explanation:", repair["explanation"])
            print("Confidence:", repair["confidence"])

            # Save the validated candidate
            solution.write_text(
                repair["fixed_code"],
                encoding="utf-8"
            )

            print("Corrected code saved.")

        print("\nMAXIMUM REPAIR ATTEMPTS REACHED.")
        print(json.dumps({
            "case_id": args.case,
            "status": "FAIL",
            "repair_attempts": attempts,
            "message": "Retry limit reached."
        }, indent=2))
        return 1

    except Exception as error:
        print(json.dumps({
            "case_id": args.case,
            "status": "ERROR",
            "message": str(error)
        }, indent=2))
        return 2


if __name__ == "__main__":
    raise SystemExit(main())
'''

AGENT_FILE.write_text(complete_code, encoding="utf-8")

print("SUCCESS: Complete debugging_agent.py updated.")
print("Backup:", PROJECT_DIR / "debugging_agent_backup.py")
print("Agent file:", AGENT_FILE)


SUCCESS: Complete debugging_agent.py updated.
Backup: /content/agentic-ai-debugging-test-automation/debugging_agent_backup.py
Agent file: /content/agentic-ai-debugging-test-automation/debugging_agent.py


In [441]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC04

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC04
Maximum Repairs: 3

Test Run 1
Exit Code: 2

==================================== ERRORS ====================================
______________________ ERROR collecting test_solution.py _______________________
/usr/local/lib/python3.13/dist-packages/_pytest/python.py:498: in importtestmodule
    mod = import_path(
/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:587: in import_path
    importlib.import_module(module_name)
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<frozen importlib._bootstrap>:1404: in _gcd_import
    ???
<frozen importlib._bootstrap>:1369: in _find_and_load
    ???
<frozen importlib._bootstrap>:1340: in _find_and_load_unlocked
    ???
<frozen importlib._bootstrap>:935: in _load_unlocked
    ???
/usr/local/lib/python3.13/dist-p

In [442]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC05"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def calculate_total(price, quantity):
    return price + quantity
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import calculate_total

def test_normal_values():
    assert calculate_total(100, 2) == 200

def test_zero_quantity():
    assert calculate_total(100, 0) == 0

def test_different_values():
    assert calculate_total(50, 3) == 150
""")

print("TC05 created successfully.")

TC05 created successfully.


In [443]:
%cd /content/agentic-ai-debugging-test-automation

!python debugging_agent.py --case TC05

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC05
Maximum Repairs: 3

Test Run 1
Exit Code: 1
FFF                                                                      [100%]
=================================== FAILURES ===================================
______________________________ test_normal_values ______________________________
test_solution.py:5: in test_normal_values
    assert calculate_total(100, 2) == 200
E   assert 102 == 200
E    +  where 102 = calculate_total(100, 2)
______________________________ test_zero_quantity ______________________________
test_solution.py:8: in test_zero_quantity
    assert calculate_total(100, 0) == 0
E   assert 100 == 0
E    +  where 100 = calculate_total(100, 0)
____________________________ test_different_values _____________________________
test_solution.py:11: in test_different_values
    assert calculate_total(50, 3) == 150
E   assert 53 == 150
E    +  where 53 = calculate_total(50, 3)
===============

In [444]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC06"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def get_last_element(numbers):
    return numbers[len(numbers)]
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import get_last_element

def test_normal_list():
    assert get_last_element([10, 20, 30]) == 30

def test_two_elements():
    assert get_last_element([5, 8]) == 8

def test_single_element():
    assert get_last_element([100]) == 100
""")

print("TC06 created successfully.")

TC06 created successfully.


In [445]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC06

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC06
Maximum Repairs: 3

Test Run 1
Exit Code: 1
                                    [100%]
=================================== FAILURES ===================================
_______________________________ test_normal_list _______________________________
test_solution.py:5: in test_normal_list
    assert get_last_element([10, 20, 30]) == 30
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in get_last_element
    return numbers[len(numbers)]
           ^^^^^^^^^^^^^^^^^^^^^
E   IndexError: list index out of range
______________________________ test_two_elements _______________________________
test_solution.py:8: in test_two_elements
    assert get_last_element([5, 8]) == 8
           ^^^^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in get_last_element
    return numbers[len(numbers)]
           ^^^^^^^^^^^^^^^^^^^^^
E   IndexError: list index out of range
_____________________________ test_single_element

In [446]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC07"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def get_student_name(student):
    return student["name"]
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import get_student_name

def test_student_with_name():
    assert get_student_name({"name": "Ravi", "age": 21}) == "Ravi"

def test_student_with_only_age():
    assert get_student_name({"age": 20}) == "Unknown"

def test_empty_student():
    assert get_student_name({}) == "Unknown"
""")

print("TC07 created successfully.")

TC07 created successfully.


In [447]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC07

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC07
Maximum Repairs: 3

Test Run 1
Exit Code: 1
.FF                                                                      [100%]
=================================== FAILURES ===================================
__________________________ test_student_with_only_age __________________________
test_solution.py:8: in test_student_with_only_age
    assert get_student_name({"age": 20}) == "Unknown"
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in get_student_name
    return student["name"]
           ^^^^^^^^^^^^^^^
E   KeyError: 'name'
______________________________ test_empty_student ______________________________
test_solution.py:11: in test_empty_student
    assert get_student_name({}) == "Unknown"
           ^^^^^^^^^^^^^^^^^^^^
solution.py:3: in get_student_name
    return student["name"]
           ^^^^^^^^^^^^^^^
E   KeyError: 'name'
=========================== short test summary info =====

In [448]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC08"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def safe_divide(a, b):
    return a / b
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import safe_divide

def test_normal_division():
    assert safe_divide(10, 2) == 5

def test_zero_numerator():
    assert safe_divide(0, 5) == 0

def test_zero_denominator():
    assert safe_divide(10, 0) == 0
""")

print("TC08 created successfully.")

TC08 created successfully.


In [449]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC08

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC08
Maximum Repairs: 3

Test Run 1
Exit Code: 1
..F                                                                      [100%]
=================================== FAILURES ===================================
____________________________ test_zero_denominator _____________________________
test_solution.py:11: in test_zero_denominator
    assert safe_divide(10, 0) == 0
           ^^^^^^^^^^^^^^^^^^
solution.py:3: in safe_divide
    return a / b
           ^^^^^
E   ZeroDivisionError: division by zero
=========================== short test summary info ============================
FAILED test_solution.py::test_zero_denominator - ZeroDivisionError: division ...
1 failed, 2 passed in 0.04s


Repair Attempt 1/3
Bug Type: ZeroDivisionError
Explanation: The function attempts to divide the numerator 'a' by the denominator 'b' without checking if 'b' is zero. When 'b' is 0, Python raises a ZeroDivisionError. 

In [450]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC09"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def convert_to_upper(text):
    return text.uppercase()
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import convert_to_upper

def test_lowercase():
    assert convert_to_upper("hello") == "HELLO"

def test_mixed_case():
    assert convert_to_upper("Hello World") == "HELLO WORLD"

def test_empty_string():
    assert convert_to_upper("") == ""
""")

print("TC09 created successfully.")

TC09 created successfully.


In [451]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC09

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC09
Maximum Repairs: 3

Test Run 1
Exit Code: 1
0%]
=================================== FAILURES ===================================
________________________________ test_lowercase ________________________________
test_solution.py:5: in test_lowercase
    assert convert_to_upper("hello") == "HELLO"
           ^^^^^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in convert_to_upper
    return text.uppercase()
           ^^^^^^^^^^^^^^
E   AttributeError: 'str' object has no attribute 'uppercase'
_______________________________ test_mixed_case ________________________________
test_solution.py:8: in test_mixed_case
    assert convert_to_upper("Hello World") == "HELLO WORLD"
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in convert_to_upper
    return text.uppercase()
           ^^^^^^^^^^^^^^
E   AttributeError: 'str' object has no attribute 'uppercase'
______________________________ test_empty_string __

In [452]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC10"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def find_maximum(numbers):
    return min(numbers)
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import find_maximum

def test_positive_numbers():
    assert find_maximum([10, 20, 30, 5]) == 30

def test_negative_numbers():
    assert find_maximum([-10, -5, -20]) == -5

def test_mixed_numbers():
    assert find_maximum([-5, 0, 15, 8]) == 15
""")

print("TC10 created successfully.")

TC10 created successfully.


In [453]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC10

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC10
Maximum Repairs: 3

Test Run 1
Exit Code: 1
FFF                                                                      [100%]
=================================== FAILURES ===================================
____________________________ test_positive_numbers _____________________________
test_solution.py:5: in test_positive_numbers
    assert find_maximum([10, 20, 30, 5]) == 30
E   assert 5 == 30
E    +  where 5 = find_maximum([10, 20, 30, 5])
____________________________ test_negative_numbers _____________________________
test_solution.py:8: in test_negative_numbers
    assert find_maximum([-10, -5, -20]) == -5
E   assert -20 == -5
E    +  where -20 = find_maximum([-10, -5, -20])
______________________________ test_mixed_numbers ______________________________
test_solution.py:11: in test_mixed_numbers
    assert find_maximum([-5, 0, 15, 8]) == 15
E   assert -5 == 15
E    +  where -5 = find_maximum(

In [454]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC11"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def calculate_sum(n):
    total = 0
    for i in range(1, n):
        total += i
    return total
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import calculate_sum

def test_small_number():
    assert calculate_sum(5) == 15

def test_single_number():
    assert calculate_sum(1) == 1

def test_larger_number():
    assert calculate_sum(10) == 55
""")

print("TC11 created successfully.")

TC11 created successfully.


In [455]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC11

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC11
Maximum Repairs: 3

Test Run 1
Exit Code: 1
FFF                                                                      [100%]
=================================== FAILURES ===================================
______________________________ test_small_number _______________________________
test_solution.py:5: in test_small_number
    assert calculate_sum(5) == 15
E   assert 10 == 15
E    +  where 10 = calculate_sum(5)
______________________________ test_single_number ______________________________
test_solution.py:8: in test_single_number
    assert calculate_sum(1) == 1
E   assert 0 == 1
E    +  where 0 = calculate_sum(1)
______________________________ test_larger_number ______________________________
test_solution.py:11: in test_larger_number
    assert calculate_sum(10) == 55
E   assert 45 == 55
E    +  where 45 = calculate_sum(10)
=========================== short test summary info ===============

In [456]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC12"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def clean_text(text):
    return text.replace(" ", "")
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import clean_text

def test_normal_text():
    assert clean_text("hello world") == "hello world"

def test_multiple_spaces():
    assert clean_text("hello   world") == "hello world"

def test_leading_trailing_spaces():
    assert clean_text("  hello  ") == "hello"
""")

print("TC12 created successfully.")

TC12 created successfully.


In [457]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC12

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC12
Maximum Repairs: 3

Test Run 1
Exit Code: 1
FF.                                                                      [100%]
=================================== FAILURES ===================================
_______________________________ test_normal_text _______________________________
test_solution.py:5: in test_normal_text
    assert clean_text("hello world") == "hello world"
E   AssertionError: assert 'helloworld' == 'hello world'
E     
E     - hello world
E     ?      -
E     + helloworld
_____________________________ test_multiple_spaces _____________________________
test_solution.py:8: in test_multiple_spaces
    assert clean_text("hello   world") == "hello world"
E   AssertionError: assert 'helloworld' == 'hello world'
E     
E     - hello world
E     ?      -
E     + helloworld
=========================== short test summary info ============================
FAILED test_solution.py::test_n

In [458]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC13"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def factorial(n):
    if n == 1:
        return 1
    return n * factorial(n - 1)
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import factorial

def test_factorial_positive():
    assert factorial(5) == 120

def test_factorial_one():
    assert factorial(1) == 1

def test_factorial_zero():
    assert factorial(0) == 1
""")

print("TC13 created successfully.")

TC13 created successfully.


In [459]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC13

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC13
Maximum Repairs: 3

Test Run 1
Exit Code: 1
 return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
               ^^^^^^^^^^^^^^^^
solution.py:5: in factorial
    return n * factorial(n - 1)
        

In [460]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC14"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def calculate_square_root(number):
    return number ** 0.5
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import calculate_square_root
import pytest

def test_positive_number():
    assert calculate_square_root(25) == 5

def test_zero():
    assert calculate_square_root(0) == 0

def test_negative_number():
    with pytest.raises(ValueError):
        calculate_square_root(-9)
""")

print("TC14 created successfully.")

TC14 created successfully.


In [461]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC14

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC14
Maximum Repairs: 3

Test Run 1
Exit Code: 1
..F                                                                      [100%]
=================================== FAILURES ===================================
_____________________________ test_negative_number _____________________________
test_solution.py:12: in test_negative_number
    with pytest.raises(ValueError):
         ^^^^^^^^^^^^^^^^^^^^^^^^^
E   Failed: DID NOT RAISE <class 'ValueError'>
=========================== short test summary info ============================
FAILED test_solution.py::test_negative_number - Failed: DID NOT RAISE <class ...
1 failed, 2 passed in 0.04s


Repair Attempt 1/3
Bug Type: Missing input validation
Explanation: The function calculates the square root of a number using the exponent operator `** 0.5`. When a negative number is passed, Python returns a complex number (e.g., `1j` for `-1`) instead of raising a `V

In [462]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT_DIR / "evaluation_cases" / "TC15"
CASE_DIR.mkdir(parents=True, exist_ok=True)

(CASE_DIR / "solution.py").write_text("""
def calculate_discount(price, discount):
    discounted_amount = price * discount / 100
    return price + discounted_amount
""")

(CASE_DIR / "test_solution.py").write_text("""
from solution import calculate_discount

def test_normal_discount():
    assert calculate_discount(1000, 10) == 900

def test_half_discount():
    assert calculate_discount(500, 50) == 250

def test_no_discount():
    assert calculate_discount(800, 0) == 800
""")

print("TC15 created successfully.")

TC15 created successfully.


In [463]:
%cd /content/agentic-ai-debugging-test-automation
!python debugging_agent.py --case TC15

/content/agentic-ai-debugging-test-automation

AGENTIC AI DEBUGGING SYSTEM
Test Case: TC15
Maximum Repairs: 3

Test Run 1
Exit Code: 1
FF.                                                                      [100%]
=================================== FAILURES ===================================
_____________________________ test_normal_discount _____________________________
test_solution.py:5: in test_normal_discount
    assert calculate_discount(1000, 10) == 900
E   assert 1100.0 == 900
E    +  where 1100.0 = calculate_discount(1000, 10)
______________________________ test_half_discount ______________________________
test_solution.py:8: in test_half_discount
    assert calculate_discount(500, 50) == 250
E   assert 750.0 == 250
E    +  where 750.0 = calculate_discount(500, 50)
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_discount - assert 1100.0 == 900
FAILED test_solution.py::test_half_discount - assert 750.0 == 

In [464]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")

evaluation_data = [
    ["TC01", "Off-by-one error", "3", 4, 1, "PASS"],
    ["TC02", "Incorrect conditional logic", "1", 3, 1, "PASS"],
    ["TC03", "NameError", "3", 3, 1, "PASS"],
    ["TC04", "SyntaxError", "SyntaxError during initial test", 3, 1, "PASS"],
    ["TC05", "Arithmetic operation error", "3", 3, 1, "PASS"],
    ["TC06", "IndexError", "3", 3, 1, "PASS"],
    ["TC07", "KeyError", "2", 3, 1, "PASS"],
    ["TC08", "ZeroDivisionError", "1", 3, 1, "PASS"],
    ["TC09", "AttributeError", "3", 3, 1, "PASS"],
    ["TC10", "Logical error", "3", 3, 1, "PASS"],
    ["TC11", "Loop boundary error", "3", 3, 1, "PASS"],
    ["TC12", "String handling error", "2", 3, 1, "PASS"],
    ["TC13", "Recursion error", "1", 3, 1, "PASS"],
    ["TC14", "Missing input validation", "1", 3, 1, "PASS"],
    ["TC15", "Discount calculation error", "2", 3, 1, "PASS"]
]

columns = [
    "case_id",
    "bug_type",
    "initial_failures",
    "final_tests_passed",
    "repair_attempts",
    "status"
]

df = pd.DataFrame(evaluation_data, columns=columns)

CSV_PATH = PROJECT_DIR / "week2_evaluation_results.csv"
df.to_csv(CSV_PATH, index=False)

print("Evaluation CSV created successfully!")
print("Saved at:", CSV_PATH)
display(df)

print("\nTotal test cases:", len(df))
print("Passed cases:", (df["status"] == "PASS").sum())
print("Failed cases:", (df["status"] == "FAIL").sum())
print("Total repair attempts:", df["repair_attempts"].sum())

Evaluation CSV created successfully!
Saved at: /content/agentic-ai-debugging-test-automation/week2_evaluation_results.csv


,case_id,bug_type,initial_failures,final_tests_passed,repair_attempts,status
0,TC01,Off-by-one error,3,4,1,PASS
1,TC02,Incorrect conditional logic,1,3,1,PASS
2,TC03,NameError,3,3,1,PASS
3,TC04,SyntaxError,SyntaxError during initial test,3,1,PASS
4,TC05,Arithmetic operation error,3,3,1,PASS
5,TC06,IndexError,3,3,1,PASS
6,TC07,KeyError,2,3,1,PASS
7,TC08,ZeroDivisionError,1,3,1,PASS
8,TC09,AttributeError,3,3,1,PASS
9,TC10,Logical error,3,3,1,PASS



Total test cases: 15
Passed cases: 15
Failed cases: 0
Total repair attempts: 15


In [465]:
from pathlib import Path

CSV_PATH = Path(
    "/content/agentic-ai-debugging-test-automation/week2_evaluation_results.csv"
)

print("File exists:", CSV_PATH.exists())
print("File size:", CSV_PATH.stat().st_size, "bytes")

File exists: True
File size: 617 bytes


In [466]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")

print("PROJECT FILE VERIFICATION")
print("=========================")

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file() and "__pycache__" not in str(path):
        print(path.relative_to(PROJECT_DIR))

print("\nImportant file checks:")

required_files = [
    "debugging_agent.py",
    "agent_cli.py",
    "README.md",
    "week2_evaluation_results.csv",
]

for filename in required_files:
    print(filename, ":", "FOUND" if (PROJECT_DIR / filename).exists() else "MISSING")

print("app.py :", "FOUND" if (PROJECT_DIR / "app.py").exists() else "MISSING")

PROJECT FILE VERIFICATION
.gitignore
.pytest_cache/.gitignore
.pytest_cache/CACHEDIR.TAG
.pytest_cache/README.md
.pytest_cache/v/cache/lastfailed
.pytest_cache/v/cache/nodeids
README.md
agent_cli.py
agentic-ai-debugging-test-automation/.git/HEAD
agentic-ai-debugging-test-automation/.git/config
agentic-ai-debugging-test-automation/.git/description
agentic-ai-debugging-test-automation/.git/hooks/applypatch-msg.sample
agentic-ai-debugging-test-automation/.git/hooks/commit-msg.sample
agentic-ai-debugging-test-automation/.git/hooks/fsmonitor-watchman.sample
agentic-ai-debugging-test-automation/.git/hooks/post-update.sample
agentic-ai-debugging-test-automation/.git/hooks/pre-applypatch.sample
agentic-ai-debugging-test-automation/.git/hooks/pre-commit.sample
agentic-ai-debugging-test-automation/.git/hooks/pre-merge-commit.sample
agentic-ai-debugging-test-automation/.git/hooks/pre-push.sample
agentic-ai-debugging-test-automation/.git/hooks/pre-rebase.sample
agentic-ai-debugging-test-automation

In [467]:
%cd /content/agentic-ai-debugging-test-automation

!streamlit --version

/content/agentic-ai-debugging-test-automation
Streamlit, version 1.66.1


In [468]:
%cd /content/agentic-ai-debugging-test-automation

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &

!sleep 5

from google.colab import output
output.serve_kernel_port_as_window(8501)

/content/agentic-ai-debugging-test-automation


2026-11-02 10:57:27.291 Port 8501 is not available
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [471]:
from google.colab import output
output.serve_kernel_port_as_iframe(8502, height=800)

<IPython.core.display.Javascript object>

In [472]:
import requests
import subprocess

print("CHECKING STREAMLIT SERVER")
print("=========================")

try:
    response = requests.get("http://127.0.0.1:8502", timeout=10)
    print("Server status:", response.status_code)
    print("Server responding:", response.status_code == 200)
except Exception as e:
    print("Server error:", e)

print("\nSTREAMLIT PROCESS:")
!ps aux | grep '[s]treamlit'

CHECKING STREAMLIT SERVER
Server error: HTTPConnectionPool(host='127.0.0.1', port=8502): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d6018449e80>: Failed to establish a new connection: [Errno 111] Connection refused'))

STREAMLIT PROCESS:


In [473]:
import subprocess
import time
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")
LOG_FILE = "/content/streamlit_log.txt"

log = open(LOG_FILE, "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    cwd=PROJECT_DIR,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(10)

print("Streamlit process status:", process.poll())
print("\nSERVER LOG:")
print(Path(LOG_FILE).read_text())

Streamlit process status: None

SERVER LOG:


2026-11-02 11:04:26.854 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.26.183.139:8502




In [474]:
import requests
from google.colab import output

response = requests.get("http://127.0.0.1:8502", timeout=15)

print("HTTP Status:", response.status_code)

output.serve_kernel_port_as_iframe(8502, height=800)

HTTP Status: 200


<IPython.core.display.Javascript object>

In [475]:
from pathlib import Path
import time

time.sleep(5)

print("LATEST STREAMLIT LOGS")
print("=====================")

log_path = Path("/content/streamlit_log.txt")
print(log_path.read_text()[-6000:])

LATEST STREAMLIT LOGS
Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8502-m-s-use1b10-3ncc0aujta5en-b.us-east1.prod.r.colab.dev, host=m-s-use1b10-3ncc0aujta5en.us-east1-b.c.codatalab-user-runtimes.internal:8007
2026-11-02 11:05:15.954 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8502-m-s-use1b10-3ncc0aujta5en-b.us-east1.prod.r.colab.dev, host=m-s-use1b10-3ncc0aujta5en.us-east1-b.c.codatalab-user-runtimes.internal:8007
2026-11-02 11:05:17.645 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8502-m-s-use1b10-3ncc0aujta5en-b.us-east1.prod.r.colab.dev, host=m-s-use1b10-3ncc0aujta5en.us-east1-b.c.codatalab-user-runtimes.internal:8007
2026-11-02 11:05:25.411 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8502-m-s-use1b10-3ncc0aujta5en-b.us-east1.prod.r.colab.dev, host=m-s-use1b10-3ncc0aujta5en.us-east1-b.c.codatalab-user-runtimes.internal:800

In [476]:
import subprocess
import psutil
import time
from pathlib import Path

# Stop the old Streamlit process
for process in psutil.process_iter(["pid", "cmdline"]):
    try:
        cmd = process.info["cmdline"] or []
        if "streamlit" in " ".join(cmd) and "app.py" in " ".join(cmd):
            process.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

time.sleep(2)

# Start Streamlit with Colab-compatible security settings
PROJECT_DIR = "/content/agentic-ai-debugging-test-automation"

log = open("/content/streamlit_log.txt", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    cwd=PROJECT_DIR,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(8)

print("Streamlit running:", process.poll() is None)
print(Path("/content/streamlit_log.txt").read_text()[-3000:])

Streamlit running: True


2026-11-02 11:08:07.046 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.26.183.139:8502




In [477]:
from google.colab import output

output.serve_kernel_port_as_iframe(8502, height=800)

<IPython.core.display.Javascript object>

In [478]:
from pathlib import Path

PROJECT_DIR = Path("/content/agentic-ai-debugging-test-automation")

print((PROJECT_DIR / "app.py").read_text())


import streamlit as st
import subprocess
import sys
import os
import difflib
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

st.set_page_config(
    page_title="Agentic AI Debugging System",
    page_icon="🐞",
    layout="wide"
)

st.title("🐞 Agentic AI Debugging & Test Automation")
st.caption("AI-assisted bug detection, code repair, and automated test validation")

st.info(
    "Select a test case and run the existing debugging agent. "
    "The agent may modify the selected case's solution file while repairing it."
)

case_id = st.selectbox(
    "Select a test case",
    ["TC01", "TC02", "TC03"]
)

case_dir = PROJECT / "evaluation_cases" / case_id
solution_file = case_dir / "solution.py"

if solution_file.exists():
    original_code = solution_file.read_text(encoding="utf-8")
    with st.expander("View current solution code"):
        st.code(original_code, language="python")
else:
    original_code = ""
    st.error(f"Solution file not fou

In [479]:

from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")
APP_FILE = PROJECT / "app.py"

new_app_code = r'''
import streamlit as st
import subprocess
import sys
import os
import difflib
import re
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

st.set_page_config(
    page_title="Agentic AI Debugging System",
    page_icon="🐞",
    layout="wide"
)

st.title("🐞 Agentic AI Debugging & Test Automation")
st.caption("AI-assisted bug detection, code repair, and automated test validation")

st.info(
    "Select any of the 15 test cases and run the Gemini-powered "
    "debugging agent. The agent may modify the selected solution file."
)

# Discover all available test cases automatically
case_ids = sorted(
    [
        folder.name
        for folder in (PROJECT / "evaluation_cases").iterdir()
        if folder.is_dir()
        and re.fullmatch(r"TC\d{2}", folder.name)
        and (folder / "solution.py").exists()
        and (folder / "test_solution.py").exists()
    ],
    key=lambda x: int(x[2:])
)

if not case_ids:
    st.error("No valid test cases were found.")
    st.stop()

st.success(f"Available test cases: {len(case_ids)}")

case_id = st.selectbox(
    "Select a test case",
    case_ids
)

case_dir = PROJECT / "evaluation_cases" / case_id
solution_file = case_dir / "solution.py"

original_code = solution_file.read_text(encoding="utf-8")

with st.expander("View current solution code"):
    st.code(original_code, language="python")

# Retrieve Gemini key securely
def get_api_key():
    key = os.environ.get("GEMINI_API_KEY")

    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
            if key:
                os.environ["GEMINI_API_KEY"] = key
        except Exception:
            pass

    return key

if st.button("🚀 Run AI Debugging Agent", type="primary"):
    api_key = get_api_key()

    if not api_key:
        st.error(
            "Gemini API key not available. Check GEMINI_API_KEY "
            "in Google Colab Secrets."
        )
        st.stop()

    with st.spinner(
        f"Gemini is analyzing and repairing {case_id}. Please wait..."
    ):
        try:
            result = subprocess.run(
                [
                    sys.executable,
                    str(PROJECT / "agent_cli.py"),
                    "--case",
                    case_id
                ],
                cwd=str(PROJECT),
                capture_output=True,
                text=True,
                timeout=240,
                env=os.environ.copy()
            )

            output = result.stdout or ""
            errors = result.stderr or ""

            repaired_code = (
                solution_file.read_text(encoding="utf-8")
                if solution_file.exists()
                else ""
            )

            st.subheader("1. Agent execution result")

            if result.returncode == 0:
                st.success("Agent execution completed successfully.")
            elif result.returncode == 1:
                st.warning("Agent finished with a failure status.")
            else:
                st.error("Agent encountered an execution error.")

            st.subheader("2. AI analysis and pytest results")
            st.code(
                output[-12000:] if output else "No output returned.",
                language="text"
            )

            if errors.strip():
                with st.expander("View error details"):
                    st.code(errors[-6000:])

            st.subheader("3. Repaired solution code")
            st.code(repaired_code, language="python")

            st.subheader("4. Before and after code comparison")

            diff = difflib.unified_diff(
                original_code.splitlines(),
                repaired_code.splitlines(),
                fromfile="Before repair",
                tofile="After repair",
                lineterm=""
            )

            diff_text = "\n".join(diff)

            if diff_text:
                st.code(diff_text, language="diff")
            else:
                st.info("No code changes were detected.")

            st.caption(
                f"Selected test case: {case_id}. "
                "Check the agent log for its repair attempts and final status."
            )

        except subprocess.TimeoutExpired:
            st.error("Agent timed out after 4 minutes.")
        except Exception as exc:
            st.error(f"Could not run the agent: {exc}")
'''

APP_FILE.write_text(new_app_code, encoding="utf-8")

print("SUCCESS: app.py updated!")
print("File:", APP_FILE)
print("All 15 test cases will be discovered automatically.")


SUCCESS: app.py updated!
File: /content/agentic-ai-debugging-test-automation/app.py
All 15 test cases will be discovered automatically.


In [480]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

cases = sorted(
    p.name for p in (PROJECT / "evaluation_cases").iterdir()
    if p.is_dir() and (p / "solution.py").exists()
)

print("AVAILABLE TEST CASES:")
for case in cases:
    print("✅", case)

print("\nTOTAL:", len(cases))

AVAILABLE TEST CASES:
✅ TC01
✅ TC02
✅ TC03
✅ TC04
✅ TC05
✅ TC06
✅ TC07
✅ TC08
✅ TC09
✅ TC10
✅ TC11
✅ TC12
✅ TC13
✅ TC14
✅ TC15

TOTAL: 15


In [481]:
import psutil
import subprocess
import time

PROJECT = "/content/agentic-ai-debugging-test-automation"

# Stop previous Streamlit process
for p in psutil.process_iter(["pid", "cmdline"]):
    try:
        cmd = " ".join(p.info["cmdline"] or [])
        if "streamlit" in cmd and "app.py" in cmd:
            p.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

time.sleep(3)

# Start updated application
log = open("/content/streamlit_log.txt", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    cwd=PROJECT,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(8)

print("Streamlit running:", process.poll() is None)
print(open("/content/streamlit_log.txt").read()[-2000:])

Streamlit running: True


2026-11-02 11:12:01.386 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.26.183.139:8502




In [482]:
from google.colab import output

output.serve_kernel_port_as_iframe(8502, height=800)

<IPython.core.display.Javascript object>

In [483]:
from pathlib import Path
import json

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

print("TC01 CURRENT SOLUTION")
print("=====================")
print((PROJECT / "evaluation_cases/TC01/solution.py").read_text())

print("\nTC01 BACKUP")
print("===========")
backup = PROJECT / "repaired_solutions_backup/TC01_solution.py"

if backup.exists():
    print(backup.read_text())
else:
    print("Backup not found")

print("\nBUG CASES FILE")
print("==============")
bug_file = PROJECT / "bug_cases.json"

if bug_file.exists():
    print(bug_file.read_text()[:5000])

TC01 CURRENT SOLUTION
def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / len(numbers)

TC01 BACKUP
def calculate_average(numbers):
    if not numbers:
        raise ValueError("List cannot be empty")
    total = sum(numbers)
    return total / len(numbers)

BUG CASES FILE
[
  {
    "id": "TC01",
    "bug_type": "Off-by-one error",
    "function_name": "calculate_average",
    "buggy_code": "def calculate_average(numbers):\n    total = sum(numbers)\n    return total / (len(numbers) - 1)\n",
    "tests": [
      "Normal average",
      "Single-element list",
      "Negative numbers",
      "Empty-list validation"
    ]
  },
  {
    "id": "TC02",
    "bug_type": "Incorrect conditional logic",
    "function_name": "is_adult",
    "buggy_code": "def is_adult(age):\n    return age > 18\n",
    "tests": [
      "Adult age",
      "Exactly 18",
      "Under 18"
    ]
  },
  {
    "id": "TC03",
    "bu

In [484]:
import json
from pathlib import Path
import shutil

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

# Read original buggy code from bug_cases.json
with open(PROJECT / "bug_cases.json", "r") as f:
    bug_cases = json.load(f)

tc01 = next(case for case in bug_cases if case["id"] == "TC01")

solution_file = PROJECT / "evaluation_cases/TC01/solution.py"

# Preserve the currently repaired solution
demo_backup = PROJECT / "TC01_repaired_before_demo.py"
shutil.copy2(solution_file, demo_backup)

# Restore original buggy code
solution_file.write_text(tc01["buggy_code"], encoding="utf-8")

print("TC01 original bug restored successfully!")
print("\nCurrent TC01 code:")
print(solution_file.read_text())

print("\nRepaired version safely backed up at:")
print(demo_backup)

TC01 original bug restored successfully!

Current TC01 code:
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)


Repaired version safely backed up at:
/content/agentic-ai-debugging-test-automation/TC01_repaired_before_demo.py


In [485]:
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)

In [486]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")
CASES_DIR = PROJECT / "evaluation_cases"

print("VERIFYING ALL 15 TEST CASES")
print("=" * 45)

for i in range(1, 16):
    case_id = f"TC{i:02d}"
    case_dir = CASES_DIR / case_id

    solution = case_dir / "solution.py"
    tests = case_dir / "test_solution.py"

    solution_status = "FOUND" if solution.exists() else "MISSING"
    test_status = "FOUND" if tests.exists() else "MISSING"

    print(f"{case_id}: Solution: {solution_status} | Tests: {test_status}")

print("\nAGENT FILES")
for filename in ["debugging_agent.py", "agent_cli.py", "app.py"]:
    path = PROJECT / filename
    print(filename, ":", "FOUND" if path.exists() else "MISSING")

VERIFYING ALL 15 TEST CASES
TC01: Solution: FOUND | Tests: FOUND
TC02: Solution: FOUND | Tests: FOUND
TC03: Solution: FOUND | Tests: FOUND
TC04: Solution: FOUND | Tests: FOUND
TC05: Solution: FOUND | Tests: FOUND
TC06: Solution: FOUND | Tests: FOUND
TC07: Solution: FOUND | Tests: FOUND
TC08: Solution: FOUND | Tests: FOUND
TC09: Solution: FOUND | Tests: FOUND
TC10: Solution: FOUND | Tests: FOUND
TC11: Solution: FOUND | Tests: FOUND
TC12: Solution: FOUND | Tests: FOUND
TC13: Solution: FOUND | Tests: FOUND
TC14: Solution: FOUND | Tests: FOUND
TC15: Solution: FOUND | Tests: FOUND

AGENT FILES
debugging_agent.py : FOUND
agent_cli.py : FOUND
app.py : FOUND


In [487]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT / "evaluation_cases" / "TC04"

print("TC04 CURRENT SOLUTION")
print("=====================")

solution = CASE_DIR / "solution.py"

if solution.exists():
    print(solution.read_text())
else:
    print("solution.py not found")

print("\nTC04 TEST FILE")
print("==============")

test_file = CASE_DIR / "test_solution.py"

if test_file.exists():
    print(test_file.read_text())
else:
    print("test_solution.py not found")

TC04 CURRENT SOLUTION
def calculate_square(number):
    return number * number


TC04 TEST FILE

from solution import calculate_square

def test_square_positive():
    assert calculate_square(4) == 16

def test_square_zero():
    assert calculate_square(0) == 0

def test_square_negative():
    assert calculate_square(-3) == 9



In [488]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")
CASE_DIR = PROJECT / "evaluation_cases" / "TC04"

solution_file = CASE_DIR / "solution.py"

# Back up the currently repaired code
backup_file = CASE_DIR / "solution_repaired_backup.py"
backup_file.write_text(solution_file.read_text(encoding="utf-8"), encoding="utf-8")

# Restore the original buggy code
buggy_code = """def calculate_square(number)
    return number * number
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC04 original bug restored successfully!")
print("\nCurrent TC04 code:")
print(solution_file.read_text())

TC04 original bug restored successfully!

Current TC04 code:
def calculate_square(number)
    return number * number



In [490]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC05" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC05" / "solution_repaired_backup.py"

# Back up the repaired version
backup_file.write_text(solution_file.read_text(encoding="utf-8"), encoding="utf-8")

# Restore original arithmetic bug
buggy_code = """def calculate_product(a, b):
    return a + b
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC05 original bug restored!")
print(solution_file.read_text())

TC05 original bug restored!
def calculate_product(a, b):
    return a + b



In [491]:
def calculate_product(a, b):
    return a + b

In [492]:
from pathlib import Path
import json

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

# Read the current TC06 code
solution_file = PROJECT / "evaluation_cases" / "TC06" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC06" / "solution_repaired_backup.py"

# Preserve the repaired version without overwriting an existing backup
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original IndexError bug
buggy_code = """def get_last_element(numbers):
    return numbers[len(numbers)]
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC06 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC06 original bug restored!
def get_last_element(numbers):
    return numbers[len(numbers)]



In [493]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC07" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC07" / "solution_repaired_backup.py"

# Preserve repaired code
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original KeyError bug
buggy_code = """def get_user_name(user):
    return user["name"]
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC07 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC07 original bug restored!
def get_user_name(user):
    return user["name"]



In [494]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC08" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC08" / "solution_repaired_backup.py"

# Preserve the repaired version
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original ZeroDivisionError bug
buggy_code = """def divide_numbers(a, b):
    return a / b
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC08 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC08 original bug restored!
def divide_numbers(a, b):
    return a / b



In [495]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC09" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC09" / "solution_repaired_backup.py"

# Preserve the repaired version
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original AttributeError bug
buggy_code = """def convert_to_uppercase(text):
    return text.uppercase()
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC09 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC09 original bug restored!
def convert_to_uppercase(text):
    return text.uppercase()



In [496]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC10" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC10" / "solution_repaired_backup.py"

# Preserve the repaired version
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original logical error
buggy_code = """def find_largest(numbers):
    return min(numbers)
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC10 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC10 original bug restored!
def find_largest(numbers):
    return min(numbers)



In [497]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC11" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC11" / "solution_repaired_backup.py"

# Preserve the repaired version
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original loop boundary error
buggy_code = """def calculate_sum(n):
    total = 0
    for i in range(1, n):
        total += i
    return total
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC11 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC11 original bug restored!
def calculate_sum(n):
    total = 0
    for i in range(1, n):
        total += i
    return total



In [498]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC12" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC12" / "solution_repaired_backup.py"

# Preserve the repaired version
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original string handling error
buggy_code = """def normalize_text(text):
    return text.replace(" ", "")
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC12 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC12 original bug restored!
def normalize_text(text):
    return text.replace(" ", "")



In [499]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC13" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC13" / "solution_repaired_backup.py"

# Preserve the repaired version
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original recursion bug
buggy_code = """def factorial(n):
    if n == 1:
        return 1
    return n * factorial(n - 1)
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC13 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC13 original bug restored!
def factorial(n):
    if n == 1:
        return 1
    return n * factorial(n - 1)



In [500]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC14" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC14" / "solution_repaired_backup.py"

# Preserve the repaired version
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original input validation bug
buggy_code = """import math

def calculate_square_root(number):
    return math.sqrt(number)
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC14 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC14 original bug restored!
import math

def calculate_square_root(number):
    return math.sqrt(number)



In [501]:
from pathlib import Path

PROJECT = Path("/content/agentic-ai-debugging-test-automation")

solution_file = PROJECT / "evaluation_cases" / "TC15" / "solution.py"
backup_file = PROJECT / "evaluation_cases" / "TC15" / "solution_repaired_backup.py"

# Preserve the repaired version
if not backup_file.exists():
    backup_file.write_text(
        solution_file.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Restore the original discount calculation error
buggy_code = """def calculate_discount(price, discount):
    return price + (price * discount / 100)
"""

solution_file.write_text(buggy_code, encoding="utf-8")

print("TC15 original bug restored!")
print(solution_file.read_text(encoding="utf-8"))

TC15 original bug restored!
def calculate_discount(price, discount):
    return price + (price * discount / 100)



In [502]:
from google.colab.output import eval_js
from IPython.display import display, HTML

url = eval_js("google.colab.kernel.proxyPort(8502)")

display(HTML(
    f'<a href="{url}" target="_blank" '
    'style="font-size:18px;">Open Agentic AI Debugging App</a>'
))